# ATLAS-FN 02 — Checkpoint Training FINAL v3: BMC seed-locked + optimised Find ablation

This notebook trains the BMC-locked checkpoint and, separately, explicitly labelled **optimised Find** checkpoints. The global seed is defined in the first user-control cell and propagated through Python, NumPy, PyTorch, Ultralytics training calls, ablation metadata, and checkpoint export manifests. The BMC checkpoint remains the manuscript reproduction model; optimised checkpoints are new research artefacts and must not be silently substituted into manuscript-locked tables.


In [ ]:
# ============================================================
# 0. USER CONTROLS — START HERE
# ============================================================
# This notebook restores the exported data-prep workspace created by the
# strict DockerRoot data-preparation notebook, then trains/export checkpoints.
#
# v4 recommended workflow:
#   1) Put the data-prep archive in Google Drive.
#   2) Paste its exact path into WORKSPACE_ARCHIVE below.
#   3) Run cells from the top.
#
# Your data-prep exporter reported:
#   archive_path: /content/atlas_fn_exports/ATLAS_FN_dataprep_prepared_plus_audit_20260705_091610.tar.gz
#   SHA256:      0a8c74561f8b289ea93b1c9bfda529c74a4d93c56a416b0e60a9edf882a91caa
#
# You indicated the Google Drive path is:
#   /content/drive/MyDrive/ATLAS_FN_exports/ATLAS_FN_dataprep_prepared_plus_audit_20260705_091610.tar
#
# This v4 notebook accepts either .tar or .tar.gz and will try obvious filename variants
# if the specified path is missing.

# Manual archive path: this is now the primary restore mechanism.
WORKSPACE_ARCHIVE = "/content/atlas_fn_exports/ATLAS_FN_dataprep_prepared_plus_audit_20260707_003119.tar.gz"

# Restore mode.
#   "manual"  = use WORKSPACE_ARCHIVE directly; no recursive Drive search.
#   "gdrive"  = search Google Drive roots only if manual path is empty/missing.
#   "content" = search /content fallback roots.
RESTORE_ARCHIVE_SOURCE = "manual"

# Google Drive controls.
GDRIVE_MOUNT_POINT = "/content/drive"
GDRIVE_FORCE_REMOUNT = False

# Fallback search is OFF by default because recursive MyDrive search can be very slow.
ENABLE_FALLBACK_ARCHIVE_SEARCH = False
GDRIVE_SEARCH_ROOTS = [
    "/content/drive/MyDrive/ATLAS_FN_exports",
]
GDRIVE_ARCHIVE_PATTERNS = [
    "ATLAS_FN_dataprep_*.tar.gz",
    "ATLAS_FN_dataprep_*.tgz",
    "ATLAS_FN_dataprep_*.tar",
    "ATLAS_FN_dataprep_*.zip",
]
GDRIVE_ARCHIVE_INDEX = None
AUTO_SELECT_LATEST_GDRIVE_ARCHIVE = True

# Optional checksum verification. Keep this set for the known archive.
EXPECTED_WORKSPACE_ARCHIVE_SHA256 = "0a8c74561f8b289ea93b1c9bfda529c74a4d93c56a416b0e60a9edf882a91caa"

# Fallback local /content search roots, used only if ENABLE_FALLBACK_ARCHIVE_SEARCH=True.
CONTENT_ARCHIVE_SEARCH_ROOTS = [
    "/content/atlas_fn_exports",
    "/content",
]

# Archive transfer / restore progress controls.
# The selected Drive archive is staged to local /content with a visible progress bar
# before extraction. This avoids slow silent tar reads from the Drive FUSE mount.
STAGE_ARCHIVE_TO_LOCAL_CONTENT = True
LOCAL_ARCHIVE_CACHE_DIR = "/content/atlas_fn_archive_cache"
ARCHIVE_COPY_CHUNK_MB = 32
SHOW_ARCHIVE_TRANSFER_PROGRESS = True
SHOW_ARCHIVE_EXTRACT_PROGRESS = True
SHOW_SHA256_PROGRESS = True
REUSE_STAGED_ARCHIVE_IF_SIZE_MATCHES = True

# Restore repair controls.
# The data-prep exporter may have packaged root-level DockerRoot datasets
# differently from data/prepared/. This training notebook repairs that layout
# deterministically from dataset_structures + source_ledger_materialised.csv.
REPAIR_RESTORED_WORKSPACE_DATASETS = True
CREATE_DATA_PREP_ALIASES = True
RESTORE_REPAIR_OVERWRITE_EMPTY_DATASET_DIRS = True
ROI_PAD_PX = 25
ROI_CLAHE_CLIP_LIMIT = 2.0
ROI_CLAHE_TILE_GRID = (8, 8)
EXPECTED_SPLIT_FRAMES = {"train": 2629, "val": 575, "test": 586}

# Main runtime roots.
WORKSPACE = "/content/atlas_fn_workspace"
EXPORT_ROOT = "/content/atlas_fn_training_exports"

# Reproducibility seed lock.
# BMC manuscript reports seed 42 for the locked HC18 implementation and reproduction notebooks.
GLOBAL_SEED = 42
SEED = GLOBAL_SEED
RANDOM_SEED = GLOBAL_SEED
PYTHONHASHSEED = str(GLOBAL_SEED)


# Training controls.
RUN_FIND_TRAINING = True
RUN_GRANDMASTER_CONFIRM_TRAINING = True
RUN_NANO_SAM2_CONFIRM_TRAINING = True
RUN_TRUE_MEASURE_TRAINING = True

# Resume controls — use these after an interrupted Colab cell run.
RESUME_EXISTING_ARTIFACTS = True
REUSE_LAST_PT_IF_BEST_MISSING = True
FORCE_RETRAIN = False
FORCE_REBUILD_NANO_SAM2_DATASET = False
FORCE_REBUILD_MEASURE_SEG_DATASET = False

# Execution mode.
SMOKE_TEST = False  # True runs tiny epochs/subsets only for notebook debugging.
STOP_ON_CRITICAL = False  # False logs issues and skips stages where possible.

# Export controls.
COPY_EXPORT_TO_DRIVE_IF_AVAILABLE = True
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/ATLAS_FN_exports"
EXPORT_COMPRESSION = "gz"  # "gz" or "none"

# Fast file access.
YOLO_CACHE_MODE = "off"  # "off", "disk", or "ram". Use "ram" only with high RAM.

# Model assets — manuscript-correct v8.4.0 assets.
YOLO26_DET_URL = "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n.pt"
YOLO26_SEG_URL = "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n-seg.pt"
SAM2_SMALL_URL = "https://github.com/ultralytics/assets/releases/download/v8.4.0/sam2_s.pt"

print("Controls loaded. Next cell restores the exported data-prep workspace using WORKSPACE_ARCHIVE directly, with progress.")


Controls loaded. Next cell restores the exported data-prep workspace using WORKSPACE_ARCHIVE directly, with progress.


In [ ]:
# ============================================================
# 1. RUNTIME SETUP, IMPORTS, AND LOGGING
# ============================================================
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import os, sys, json, shutil, tarfile, zipfile, hashlib, time, platform, subprocess, gc, math, random, traceback

# Optional installs. Keep minimal because Colab often already has these.
def pip_install_if_missing(import_name: str, package_name: str):
    try:
        __import__(import_name)
        return False
    except Exception:
        print(f"[setup] installing {package_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        return True

pip_install_if_missing("ultralytics", "ultralytics>=8.4.0")
pip_install_if_missing("yaml", "pyyaml")

import yaml
import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from ultralytics import YOLO, SAM, settings

WORKSPACE = Path(WORKSPACE)
EXPORT_ROOT = Path(EXPORT_ROOT)
RUNS_DIR = WORKSPACE / "Training_Runs"
OUTPUT_ROOT = WORKSPACE / "outputs" / "training_checkpoint_production"
BASE_WEIGHTS = WORKSPACE / "base_weights"
DATA_PREP_DIR = WORKSPACE / "data" / "prepared"

for p in [WORKSPACE, EXPORT_ROOT, RUNS_DIR, OUTPUT_ROOT, BASE_WEIGHTS, OUTPUT_ROOT / "logs", OUTPUT_ROOT / "tables", OUTPUT_ROOT / "manifests", OUTPUT_ROOT / "figures"]:
    p.mkdir(parents=True, exist_ok=True)

# Keep tool caches local and fast.
os.environ.setdefault("ULTRALYTICS_CONFIG_DIR", str(WORKSPACE / ".cache" / "ultralytics"))
os.environ.setdefault("YOLO_CONFIG_DIR", str(WORKSPACE / ".cache" / "ultralytics"))
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".cache" / "torch"))
os.environ.setdefault("HF_HOME", str(WORKSPACE / ".cache" / "hf"))
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
settings.update({"tensorboard": False, "wandb": False, "hub": False})


# ------------------------------------------------------------
# Reproducibility seed setup — must run before any config dicts or training calls.
# ------------------------------------------------------------
def set_global_seed(seed: int = 42):
    """Set Python/NumPy/PyTorch seeds and record the deterministic runtime contract."""
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Keep deterministic algorithms disabled by default because some Ultralytics/CUDA
        # kernels are not deterministic-safe in Colab; this records the seed while preserving
        # the manuscript-compatible training path.
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    except Exception as e:
        print(f"[seed warning] PyTorch seed setup incomplete: {e!r}")
    return seed

# Backward-compatible aliases used by downstream cells.
try:
    GLOBAL_SEED
except NameError:
    GLOBAL_SEED = 42
SEED = set_global_seed(GLOBAL_SEED)
RANDOM_SEED = SEED

ISSUES = []
STAGE_AUDIT = []

def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def log(msg):
    print(f"[atlas-train] {msg}")

def record_issue(layer, severity, message, **kw):
    row = {"ts": utc_now(), "layer": layer, "severity": severity, "message": message, **kw}
    ISSUES.append(row)
    print(f"[{severity}:{layer}] {message}" + (f" | {kw}" if kw else ""))
    return row

def record_stage(stage, status, **kw):
    row = {"ts": utc_now(), "stage": stage, "status": status, **kw}
    STAGE_AUDIT.append(row)
    print(f"[stage:{status}] {stage}" + (f" | {kw}" if kw else ""))
    return row

def save_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2, ensure_ascii=False))

def sha256_file(path, block_size=1024*1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda: f.read(block_size), b""):
            h.update(b)
    return h.hexdigest()

def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024**2) if path.exists() else 0.0

def clean_system():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def maybe_raise(message):
    if STOP_ON_CRITICAL:
        raise RuntimeError(message)
    record_issue("critical_guard", "warning", message)

runtime_info = {
    "created_utc": utc_now(),
    "python": sys.version,
    "platform": platform.platform(),
    "cuda_available": torch.cuda.is_available(),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "workspace": str(WORKSPACE),
}
save_json(OUTPUT_ROOT / "manifests/runtime_info.json", runtime_info)
print(json.dumps(runtime_info, indent=2))


[setup] installing ultralytics>=8.4.0 ...
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{
  "created_utc": "2026-07-07T01:37:43+00:00",
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "cuda_available": true,
  "device_name": "NVIDIA A100-SXM4-80GB",
  "workspace": "/content/atlas_fn_workspace"
}


In [ ]:
# ============================================================
# 2. RESTORE EXPORTED DATA-PREP WORKSPACE FROM EXPLICIT GDRIVE PATH
#    No recursive Google Drive search by default; visible copy/checksum/extract progress.
# ============================================================
# This cell is intentionally direct and fast:
#   - mount Google Drive only if the manual archive path is on Drive;
#   - validate WORKSPACE_ARCHIVE immediately;
#   - try .tar/.tar.gz/.tgz variants if the exact path is missing;
#   - stage the archive to local /content with progress;
#   - verify SHA256 when EXPECTED_WORKSPACE_ARCHIVE_SHA256 is set;
#   - extract and restore the workspace with progress.

from pathlib import Path
from datetime import datetime, timezone
import tarfile, zipfile, shutil, json, os, time, hashlib
from tqdm.auto import tqdm

ARCHIVE_CANDIDATES = []
ARCHIVE_TRANSFER_RECORD = None
ARCHIVE_CHECKSUM_RECORD = None


def format_bytes(n: int) -> str:
    n = float(n or 0)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024.0:
            return f"{n:.2f} {unit}"
        n /= 1024.0
    return f"{n:.2f} PB"


def normalise_colab_path(p) -> Path:
    """Accept both '/content/...' and 'content/...' user-pasted paths."""
    s = str(p or "").strip().strip('"').strip("'")
    if s.startswith("content/"):
        s = "/" + s
    return Path(s).expanduser()


def path_is_on_gdrive(path: Path) -> bool:
    try:
        return str(Path(path).resolve()).startswith(str(Path(GDRIVE_MOUNT_POINT).resolve()))
    except Exception:
        return str(path).startswith(str(GDRIVE_MOUNT_POINT))


def mount_google_drive_if_needed_for_path(path: Path):
    """Mount Google Drive only when the explicit archive path is under /content/drive."""
    mount_point = Path(GDRIVE_MOUNT_POINT)
    if not str(path).startswith(str(mount_point)):
        return False
    if (mount_point / "MyDrive").exists():
        log(f"Google Drive already mounted at {mount_point}")
        return True
    try:
        from google.colab import drive
        log(f"mounting Google Drive at {mount_point}")
        drive.mount(str(mount_point), force_remount=GDRIVE_FORCE_REMOUNT)
        ok = (mount_point / "MyDrive").exists()
        log(f"Google Drive mount status: {'ok' if ok else 'not visible'}")
        return ok
    except Exception as e:
        record_issue("restore", "warning", "Google Drive mount unavailable", error=repr(e))
        return False


def candidate_path_variants(path: Path):
    """Try likely extension variants without scanning Drive."""
    variants = []
    s = str(path)
    variants.append(path)
    if s.endswith(".tar"):
        variants.append(Path(s + ".gz"))
        variants.append(Path(s[:-4] + ".tar.gz"))
        variants.append(Path(s[:-4] + ".tgz"))
        variants.append(Path(s[:-4] + ".zip"))
    elif s.endswith(".tar.gz"):
        variants.append(Path(s[:-3]))       # .tar
        variants.append(Path(s[:-7] + ".tgz"))
        variants.append(Path(s[:-7] + ".zip"))
    elif s.endswith(".tgz"):
        variants.append(Path(s[:-4] + ".tar.gz"))
        variants.append(Path(s[:-4] + ".tar"))
        variants.append(Path(s[:-4] + ".zip"))
    elif s.endswith(".zip"):
        variants.append(Path(s[:-4] + ".tar.gz"))
        variants.append(Path(s[:-4] + ".tar"))
        variants.append(Path(s[:-4] + ".tgz"))
    else:
        variants.extend([Path(s + ext) for ext in [".tar.gz", ".tar", ".tgz", ".zip"]])
    # preserve order and uniqueness
    out, seen = [], set()
    for v in variants:
        key = str(v)
        if key not in seen:
            out.append(v); seen.add(key)
    return out


def find_archives_under_roots(roots, patterns, recursive=False):
    """Optional fallback search. Non-recursive by default to avoid long Drive scans."""
    found = []
    seen = set()
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pattern in patterns:
            iterator = root.rglob(pattern) if recursive else root.glob(pattern)
            for p in iterator:
                try:
                    p = p.resolve()
                    if p.is_file() and p.stat().st_size > 0 and str(p) not in seen:
                        found.append(p); seen.add(str(p))
                except Exception:
                    pass
    return sorted(found, key=lambda p: (p.stat().st_mtime, p.stat().st_size, str(p)), reverse=True)


def choose_workspace_archive_direct():
    """Select archive by explicit path first; no recursive Drive search unless enabled."""
    global WORKSPACE_ARCHIVE, ARCHIVE_CANDIDATES

    manual = normalise_colab_path(WORKSPACE_ARCHIVE)
    if str(manual):
        mount_google_drive_if_needed_for_path(manual)
        variants = candidate_path_variants(manual)
        rows = []
        for i, v in enumerate(variants):
            exists = v.exists() and v.is_file()
            rows.append({
                "index": i,
                "candidate_path": str(v),
                "exists": exists,
                "size_gb": round(v.stat().st_size / (1024**3), 4) if exists else None,
                "size": format_bytes(v.stat().st_size) if exists else None,
            })
        df = pd.DataFrame(rows)
        out_csv = OUTPUT_ROOT / "tables" / "manual_archive_path_resolution.csv"
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_csv, index=False)
        print("\nManual archive path resolution")
        display(df)
        log(f"manual archive resolution table written: {out_csv}")
        for v in variants:
            if v.exists() and v.is_file() and v.stat().st_size > 0:
                WORKSPACE_ARCHIVE = str(v)
                ARCHIVE_CANDIDATES = [v]
                log(f"selected explicit workspace archive: {v} ({format_bytes(v.stat().st_size)})")
                return v
        record_issue("restore", "warning", "Explicit WORKSPACE_ARCHIVE not found after extension-variant checks", path=str(manual))

    if not ENABLE_FALLBACK_ARCHIVE_SEARCH:
        maybe_raise("No workspace archive found. Set WORKSPACE_ARCHIVE to the exact Google Drive path or enable fallback search.")
        return None

    log("fallback archive search enabled; searching only configured roots")
    candidates = []
    if RESTORE_ARCHIVE_SOURCE in {"gdrive", "manual"}:
        # Mount if any search root is on Drive.
        for r in GDRIVE_SEARCH_ROOTS:
            if str(r).startswith(str(GDRIVE_MOUNT_POINT)):
                mount_google_drive_if_needed_for_path(Path(r) / "dummy")
                break
        candidates.extend(find_archives_under_roots(GDRIVE_SEARCH_ROOTS, GDRIVE_ARCHIVE_PATTERNS, recursive=False))
    if RESTORE_ARCHIVE_SOURCE in {"content", "manual", "gdrive"}:
        candidates.extend(find_archives_under_roots(CONTENT_ARCHIVE_SEARCH_ROOTS, GDRIVE_ARCHIVE_PATTERNS, recursive=False))

    # Deduplicate.
    dedup, seen = [], set()
    for p in candidates:
        if str(p) not in seen:
            dedup.append(p); seen.add(str(p))
    candidates = dedup
    ARCHIVE_CANDIDATES = candidates
    df = pd.DataFrame([{
        "index": i,
        "path": str(p),
        "size_gb": round(p.stat().st_size / (1024**3), 4),
        "modified_utc": datetime.fromtimestamp(p.stat().st_mtime, timezone.utc).isoformat(timespec="seconds"),
    } for i, p in enumerate(candidates)])
    out_csv = OUTPUT_ROOT / "tables" / "workspace_archive_candidates.csv"
    df.to_csv(out_csv, index=False)
    print("\nFallback archive candidates")
    display(df)
    if not candidates:
        maybe_raise("Fallback search found no workspace archives.")
        return None
    idx = int(GDRIVE_ARCHIVE_INDEX) if GDRIVE_ARCHIVE_INDEX is not None else 0
    selected = candidates[idx]
    WORKSPACE_ARCHIVE = str(selected)
    log(f"selected fallback workspace archive: {selected}")
    return selected


def copy_file_with_progress(src: Path, dst: Path, desc: str = "stage archive", chunk_mb: int = 32):
    """Copy a large archive with explicit progress, speed and elapsed time."""
    global ARCHIVE_TRANSFER_RECORD
    src = Path(src); dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    total = src.stat().st_size
    chunk_size = int(max(1, chunk_mb)) * 1024 * 1024
    tmp = dst.with_suffix(dst.suffix + ".partial")

    if REUSE_STAGED_ARCHIVE_IF_SIZE_MATCHES and dst.exists() and dst.stat().st_size == total:
        log(f"reusing local staged archive: {dst} ({format_bytes(total)})")
        ARCHIVE_TRANSFER_RECORD = {
            "status": "reused_existing_local_stage", "src": str(src), "dst": str(dst),
            "bytes": total, "elapsed_seconds": 0.0, "mb_per_second": None,
        }
        save_json(OUTPUT_ROOT / "manifests/archive_transfer_manifest.json", ARCHIVE_TRANSFER_RECORD)
        return dst

    if tmp.exists():
        tmp.unlink()

    log("staging archive from Google Drive/local source to local Colab storage")
    log(f"source={src}")
    log(f"target={dst}")
    log(f"size={format_bytes(total)} | chunk={chunk_mb} MB")

    copied = 0
    t0 = time.time()
    with src.open("rb") as fsrc, tmp.open("wb") as fdst:
        pbar = tqdm(total=total, unit="B", unit_scale=True, unit_divisor=1024, desc=desc, disable=not SHOW_ARCHIVE_TRANSFER_PROGRESS)
        try:
            while True:
                buf = fsrc.read(chunk_size)
                if not buf:
                    break
                fdst.write(buf)
                copied += len(buf)
                elapsed = max(time.time() - t0, 1e-9)
                pbar.set_postfix({"MB/s": f"{(copied/(1024**2))/elapsed:.1f}"})
                pbar.update(len(buf))
        finally:
            pbar.close()

    tmp.replace(dst)
    elapsed = time.time() - t0
    mbps = (copied / (1024**2)) / max(elapsed, 1e-9)
    ARCHIVE_TRANSFER_RECORD = {
        "status": "copied", "src": str(src), "dst": str(dst), "bytes": copied,
        "elapsed_seconds": elapsed, "mb_per_second": mbps,
        "completed_utc": utc_now(),
    }
    save_json(OUTPUT_ROOT / "manifests/archive_transfer_manifest.json", ARCHIVE_TRANSFER_RECORD)
    log(f"archive staged | elapsed={elapsed:.1f}s | speed={mbps:.1f} MB/s")
    return dst


def stage_archive_to_local_if_needed(archive_path: Path):
    archive_path = Path(archive_path)
    if not STAGE_ARCHIVE_TO_LOCAL_CONTENT:
        return archive_path
    local_cache = Path(LOCAL_ARCHIVE_CACHE_DIR)
    local_cache.mkdir(parents=True, exist_ok=True)
    local_path = local_cache / archive_path.name
    # Even if source is already in /content, keep a single local-cache path for consistent restore.
    if archive_path.resolve() == local_path.resolve():
        return archive_path
    return copy_file_with_progress(archive_path, local_path, desc="stage data-prep archive", chunk_mb=ARCHIVE_COPY_CHUNK_MB)


def sha256_file_with_progress(path: Path, block_mb: int = 32):
    path = Path(path)
    total = path.stat().st_size
    block_size = int(max(1, block_mb)) * 1024 * 1024
    h = hashlib.sha256()
    read = 0
    t0 = time.time()
    with path.open("rb") as f:
        pbar = tqdm(total=total, unit="B", unit_scale=True, unit_divisor=1024, desc="SHA256 archive", disable=not SHOW_SHA256_PROGRESS)
        try:
            while True:
                b = f.read(block_size)
                if not b:
                    break
                h.update(b)
                read += len(b)
                elapsed = max(time.time() - t0, 1e-9)
                pbar.set_postfix({"MB/s": f"{(read/(1024**2))/elapsed:.1f}"})
                pbar.update(len(b))
        finally:
            pbar.close()
    return h.hexdigest()


def verify_archive_checksum_if_requested(path: Path):
    global ARCHIVE_CHECKSUM_RECORD
    path = Path(path)
    expected = str(EXPECTED_WORKSPACE_ARCHIVE_SHA256 or "").strip().lower()
    if not expected:
        ARCHIVE_CHECKSUM_RECORD = {"status": "skipped", "path": str(path)}
        save_json(OUTPUT_ROOT / "manifests/archive_checksum_manifest.json", ARCHIVE_CHECKSUM_RECORD)
        log("SHA256 verification skipped because EXPECTED_WORKSPACE_ARCHIVE_SHA256 is empty")
        return ARCHIVE_CHECKSUM_RECORD
    log("verifying staged archive SHA256")
    actual = sha256_file_with_progress(path, block_mb=ARCHIVE_COPY_CHUNK_MB).lower()
    ok = actual == expected
    ARCHIVE_CHECKSUM_RECORD = {"status": "pass" if ok else "fail", "path": str(path), "expected": expected, "actual": actual}
    save_json(OUTPUT_ROOT / "manifests/archive_checksum_manifest.json", ARCHIVE_CHECKSUM_RECORD)
    if ok:
        log("SHA256 verification PASS")
    else:
        record_issue("restore", "warning", "SHA256 verification failed; continuing only if STOP_ON_CRITICAL=False", expected=expected, actual=actual)
        if STOP_ON_CRITICAL:
            raise RuntimeError("SHA256 verification failed")
    return ARCHIVE_CHECKSUM_RECORD


def safe_extract_archive_with_progress(archive_path: Path, target_dir: Path):
    archive_path = Path(archive_path)
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    base = target_dir.resolve()
    t0 = time.time()
    extract_record = {"archive_path": str(archive_path), "target_dir": str(target_dir), "started_utc": utc_now()}

    if tarfile.is_tarfile(archive_path):
        extract_record["format"] = "tar"
        with tarfile.open(archive_path, "r:*") as tar:
            members = tar.getmembers()
            for member in members:
                dest = (target_dir / member.name).resolve()
                if not str(dest).startswith(str(base)):
                    raise RuntimeError(f"Blocked unsafe path in archive: {member.name}")
            total = int(sum(max(0, getattr(m, "size", 0) or 0) for m in members))
            extract_record["members"] = len(members)
            extract_record["bytes_declared"] = total
            log(f"extracting tar archive | members={len(members)} | declared_size={format_bytes(total)}")
            pbar = tqdm(total=total or len(members), unit="B" if total else "file", unit_scale=bool(total), unit_divisor=1024, desc="extract workspace archive", disable=not SHOW_ARCHIVE_EXTRACT_PROGRESS)
            try:
                for member in members:
                    tar.extract(member, target_dir)
                    pbar.update(max(0, getattr(member, "size", 0) or 1))
            finally:
                pbar.close()

    elif zipfile.is_zipfile(archive_path):
        extract_record["format"] = "zip"
        with zipfile.ZipFile(archive_path, "r") as zf:
            members = zf.infolist()
            for member in members:
                dest = (target_dir / member.filename).resolve()
                if not str(dest).startswith(str(base)):
                    raise RuntimeError(f"Blocked unsafe path in archive: {member.filename}")
            total = int(sum(max(0, m.file_size or 0) for m in members))
            extract_record["members"] = len(members)
            extract_record["bytes_declared"] = total
            log(f"extracting zip archive | members={len(members)} | declared_size={format_bytes(total)}")
            pbar = tqdm(total=total or len(members), unit="B" if total else "file", unit_scale=bool(total), unit_divisor=1024, desc="extract workspace archive", disable=not SHOW_ARCHIVE_EXTRACT_PROGRESS)
            try:
                for member in members:
                    zf.extract(member, target_dir)
                    pbar.update(max(0, member.file_size or 1))
            finally:
                pbar.close()
    else:
        raise ValueError(f"Unsupported or unreadable archive type: {archive_path}")

    elapsed = time.time() - t0
    extract_record["elapsed_seconds"] = elapsed
    extract_record["completed_utc"] = utc_now()
    if extract_record.get("bytes_declared"):
        extract_record["mb_per_second_declared"] = (extract_record["bytes_declared"] / (1024**2)) / max(elapsed, 1e-9)
    save_json(OUTPUT_ROOT / "manifests/archive_extraction_manifest.json", extract_record)
    log(f"archive extraction complete | elapsed={elapsed:.1f}s | target={target_dir}")
    return extract_record


def locate_extracted_export_root(restore_root: Path):
    manifest_hits = list(restore_root.rglob("EXPORT_MANIFEST.json"))
    if manifest_hits:
        return manifest_hits[0].parent
    children = [p for p in restore_root.iterdir() if p.is_dir()]
    return children[0] if len(children) == 1 else restore_root


def merge_directory(src: Path, dst: Path):
    dst.mkdir(parents=True, exist_ok=True)
    copied = 0
    total_files = sum(1 for p in src.rglob("*") if p.is_file())
    pbar = tqdm(total=total_files, desc=f"restore files: {src.name}", unit="file", disable=not SHOW_ARCHIVE_EXTRACT_PROGRESS)
    try:
        for sub in src.rglob("*"):
            rel = sub.relative_to(src)
            out = dst / rel
            if sub.is_dir():
                out.mkdir(parents=True, exist_ok=True)
            elif sub.is_file():
                out.parent.mkdir(parents=True, exist_ok=True)
                if not out.exists() or not RESUME_EXISTING_ARTIFACTS:
                    shutil.copy2(sub, out)
                    copied += 1
                pbar.update(1)
    finally:
        pbar.close()
    return copied


def restore_exported_workspace(archive_path: Path):
    if archive_path is None or not Path(archive_path).exists():
        maybe_raise("Cannot restore workspace because exported archive is missing.")
        return False

    selected_archive = Path(archive_path)
    staged_archive = stage_archive_to_local_if_needed(selected_archive)
    checksum_record = verify_archive_checksum_if_requested(Path(staged_archive))

    restore_root = Path("/content/atlas_fn_restore_stage")
    if restore_root.exists():
        shutil.rmtree(restore_root)
    restore_root.mkdir(parents=True, exist_ok=True)

    log(f"extracting exported workspace archive: {staged_archive}")
    extraction_record = safe_extract_archive_with_progress(Path(staged_archive), restore_root)

    extracted_root = locate_extracted_export_root(restore_root)
    log(f"extracted_root={extracted_root}")
    WORKSPACE.mkdir(parents=True, exist_ok=True)

    source_root = extracted_root / "atlas_fn_workspace" if (extracted_root / "atlas_fn_workspace").exists() else extracted_root

    copied = []
    for item in source_root.iterdir():
        if item.name in {"EXPORT_MANIFEST.json", "EXPORT_FILE_INVENTORY.json", "RESTORE_IN_NEXT_NOTEBOOK.py"}:
            shutil.copy2(item, WORKSPACE / item.name)
            copied.append({"item": item.name, "mode": "file"})
            continue
        dst = WORKSPACE / item.name
        if item.is_dir():
            if dst.exists() and not RESUME_EXISTING_ARTIFACTS:
                shutil.rmtree(dst)
            if dst.exists():
                n = merge_directory(item, dst)
                copied.append({"item": item.name, "mode": "merged", "files_copied": n})
            else:
                log(f"restoring directory tree: {item.name}")
                shutil.copytree(item, dst, symlinks=False)
                copied.append({"item": item.name, "mode": "copied_tree"})
        elif item.is_file():
            shutil.copy2(item, dst)
            copied.append({"item": item.name, "mode": "file"})

    restore_manifest = {
        "selected_archive": str(selected_archive),
        "staged_archive": str(staged_archive),
        "selected_archive_size_bytes": selected_archive.stat().st_size,
        "staged_archive_size_bytes": staged_archive.stat().st_size,
        "transfer_record": ARCHIVE_TRANSFER_RECORD,
        "checksum_check": checksum_record,
        "extraction_record": extraction_record,
        "extracted_root": str(extracted_root),
        "source_root": str(source_root),
        "copied": copied,
        "restored_utc": utc_now(),
        "restore_archive_source": RESTORE_ARCHIVE_SOURCE,
        "workspace_archive_control": str(WORKSPACE_ARCHIVE),
        "fallback_search_enabled": ENABLE_FALLBACK_ARCHIVE_SEARCH,
    }
    save_json(OUTPUT_ROOT / "manifests/workspace_restore_manifest.json", restore_manifest)
    log(f"workspace restored/merged into {WORKSPACE}")
    return True




# -------------------------------------------------------------------------
# Post-restore dataset layout repair
# -------------------------------------------------------------------------
# Why this exists:
#   The verified data-prep archive can contain the DockerRoot-style datasets
#   at root level (workspace/dataset_brain, workspace/dataset_roi_enhanced_gt,
#   workspace/dataset_structures), while the training notebook also accepts
#   workspace/data/prepared/*. If an earlier export omitted a root-level dataset
#   but preserved dataset_structures + source_ledger_materialised.csv, this block
#   deterministically reconstructs the missing training datasets without returning
#   to raw Zenodo extraction.

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}


def count_images_under_dataset(root: Path):
    root = Path(root)
    out = {}
    for sp in ["train", "val", "test"]:
        img_dir = root / "images" / sp
        out[sp] = len([p for p in img_dir.glob("*") if p.suffix.lower() in IMAGE_EXTS]) if img_dir.exists() else 0
    return out


def dataset_has_expected_images(root: Path, expected=None):
    expected = expected or EXPECTED_SPLIT_FRAMES
    counts = count_images_under_dataset(root)
    return all(int(counts.get(sp, 0)) == int(expected[sp]) for sp in expected), counts


def is_empty_or_incomplete_dataset_dir(root: Path):
    ok, counts = dataset_has_expected_images(root)
    return (not root.exists()) or (not ok) or (sum(counts.values()) == 0)


def safe_remove_if_empty_or_incomplete(path: Path):
    path = Path(path)
    if not path.exists():
        return False
    if path.is_symlink():
        path.unlink()
        return True
    if path.is_dir() and RESTORE_REPAIR_OVERWRITE_EMPTY_DATASET_DIRS:
        ok, counts = dataset_has_expected_images(path)
        if (not ok) or sum(counts.values()) == 0:
            shutil.rmtree(path)
            return True
    return False


def write_yolo_yaml(root: Path, names: dict, primary_name: str = "data.yaml", extra_names=None):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    data = {"path": str(root), "train": "images/train", "val": "images/val", "test": "images/test", "names": names}
    targets = [primary_name] + list(extra_names or [])
    for fn in targets:
        (root / fn).write_text(yaml.safe_dump(data, sort_keys=False))
    return root / primary_name


def hardlink_or_copy(src: Path, dst: Path):
    src = Path(src); dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 0:
        return "exists"
    try:
        os.link(src, dst)
        return "hardlink"
    except Exception:
        shutil.copy2(src, dst)
        return "copy"


def find_existing_dataset_root(name: str):
    candidates = [WORKSPACE / name, DATA_PREP_DIR / name]
    best = None
    best_total = -1
    for root in candidates:
        counts = count_images_under_dataset(root)
        total = sum(counts.values())
        if total > best_total:
            best, best_total = root, total
    return best, best_total


def find_source_ledger_materialised():
    candidates = []
    candidates.extend((WORKSPACE / "outputs").rglob("source_ledger_materialised.csv"))
    candidates.extend(WORKSPACE.rglob("source_ledger_materialised.csv"))
    unique = []
    seen = set()
    for p in candidates:
        if p.exists() and p.is_file() and str(p) not in seen:
            unique.append(p); seen.add(str(p))
    rows = []
    for p in unique:
        try:
            df = pd.read_csv(p, nrows=5)
            full_n = sum(1 for _ in open(p, "r", encoding="utf-8", errors="ignore")) - 1
            rows.append({"path": str(p), "rows": max(0, full_n), "columns": list(df.columns)})
        except Exception as e:
            rows.append({"path": str(p), "rows": None, "error": repr(e)})
    if rows:
        pd.DataFrame(rows).to_csv(OUTPUT_ROOT / "tables" / "source_ledger_candidates.csv", index=False)
    valid = [Path(r["path"]) for r in rows if r.get("rows") and r["rows"] > 0]
    if not valid:
        return None
    # Prefer the largest ledger.
    valid = sorted(valid, key=lambda p: p.stat().st_size, reverse=True)
    return valid[0]


def resolve_structure_image(struct_root: Path, split: str, filename: str, uid: str = None, stem: str = None):
    img_dir = Path(struct_root) / "images" / split
    if not img_dir.exists():
        return None
    names = []
    if filename:
        names.append(filename)
    if uid:
        for ext in [".png", ".jpg", ".jpeg", ".bmp"]:
            names.append(str(uid) + ext)
    if stem:
        for ext in [".png", ".jpg", ".jpeg", ".bmp"]:
            names.append(str(stem) + ext)
    for name in names:
        p = img_dir / name
        if p.exists() and p.is_file():
            return p
    # Slow fallback only for missing rare rows.
    wanted_stems = {Path(x).stem for x in names if x}
    for p in img_dir.glob("*"):
        if p.suffix.lower() in IMAGE_EXTS and p.stem in wanted_stems:
            return p
    return None


def rebuild_dataset_brain_from_structures_and_ledger(struct_root: Path, brain_root: Path, ledger_csv: Path):
    struct_root = Path(struct_root); brain_root = Path(brain_root); ledger_csv = Path(ledger_csv)
    if not struct_root.exists() or not ledger_csv.exists():
        record_issue("restore_repair", "warning", "Cannot rebuild dataset_brain; missing structures root or source ledger", struct_root=str(struct_root), ledger=str(ledger_csv))
        return None
    if brain_root.exists():
        shutil.rmtree(brain_root)
    for sp in ["train", "val", "test"]:
        (brain_root / "images" / sp).mkdir(parents=True, exist_ok=True)
        (brain_root / "labels" / sp).mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(ledger_csv)
    required = {"split", "brain_xc", "brain_yc", "brain_w", "brain_h"}
    missing = sorted(required - set(df.columns))
    if missing:
        record_issue("restore_repair", "warning", "Cannot rebuild dataset_brain; ledger columns missing", missing=missing, ledger=str(ledger_csv))
        return None
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc="repair dataset_brain", unit="img"):
        split = str(r.get("split", ""))
        if split not in {"train", "val", "test"}:
            continue
        filename = str(r.get("filename", "") or Path(str(r.get("source_image", ""))).name)
        uid = str(r.get("uid", "") or Path(filename).stem)
        src = resolve_structure_image(struct_root, split, filename=filename, uid=uid, stem=Path(filename).stem)
        if src is None:
            record_issue("restore_repair", "warning", "Could not resolve structure image for brain rebuild", split=split, filename=filename, uid=uid)
            continue
        dst_img = brain_root / "images" / split / src.name
        mode = hardlink_or_copy(src, dst_img)
        stem = dst_img.stem
        dst_lbl = brain_root / "labels" / split / f"{stem}.txt"
        try:
            vals = [float(r["brain_xc"]), float(r["brain_yc"]), float(r["brain_w"]), float(r["brain_h"])]
            dst_lbl.write_text("0 " + " ".join(f"{v:.6f}" for v in vals) + "\n")
            rows.append({"split": split, "image": str(dst_img), "label": str(dst_lbl), "mode": mode})
        except Exception as e:
            record_issue("restore_repair", "warning", "Could not write brain label", filename=filename, error=repr(e))
    write_yolo_yaml(brain_root, {0: "Brain"}, primary_name="data_brain.yaml", extra_names=["data.yaml"])
    man = pd.DataFrame(rows)
    man.to_csv(OUTPUT_ROOT / "tables" / "repaired_dataset_brain_manifest.csv", index=False)
    ok, counts = dataset_has_expected_images(brain_root)
    log(f"dataset_brain repair complete | ok={ok} | counts={counts}")
    return brain_root


def read_yolo_box_first(txt: Path):
    if not Path(txt).exists():
        return None
    for line in Path(txt).read_text().splitlines():
        parts = line.split()
        if len(parts) >= 5:
            try:
                return [float(x) for x in parts[1:5]]
            except Exception:
                return None
    return None


def rebuild_roi_enhanced_from_structures_and_brain(struct_root: Path, brain_root: Path, roi_root: Path):
    struct_root = Path(struct_root); brain_root = Path(brain_root); roi_root = Path(roi_root)
    if not struct_root.exists() or not brain_root.exists():
        record_issue("restore_repair", "warning", "Cannot rebuild ROI dataset; missing structures or brain root", struct_root=str(struct_root), brain_root=str(brain_root))
        return None
    if roi_root.exists():
        shutil.rmtree(roi_root)
    for sp in ["train", "val", "test"]:
        (roi_root / "images" / sp).mkdir(parents=True, exist_ok=True)
        (roi_root / "labels" / sp).mkdir(parents=True, exist_ok=True)
    manifest = []
    clahe_obj = cv2.createCLAHE(clipLimit=ROI_CLAHE_CLIP_LIMIT, tileGridSize=ROI_CLAHE_TILE_GRID)
    for split in ["train", "val", "test"]:
        src_img_dir = struct_root / "images" / split
        src_lbl_dir = struct_root / "labels" / split
        brain_lbl_dir = brain_root / "labels" / split
        img_files = [p for p in sorted(src_img_dir.glob("*")) if p.suffix.lower() in IMAGE_EXTS] if src_img_dir.exists() else []
        for img_p in tqdm(img_files, desc=f"repair ROI {split}", unit="img"):
            stem = img_p.stem
            im = cv2.imread(str(img_p))
            if im is None:
                record_issue("restore_repair", "warning", "Could not read image for ROI repair", image=str(img_p))
                continue
            H, W = im.shape[:2]
            b = read_yolo_box_first(brain_lbl_dir / f"{stem}.txt")
            if b:
                xc, yc, bw, bh = b
                x1, y1 = int((xc - bw / 2) * W), int((yc - bh / 2) * H)
                x2, y2 = int((xc + bw / 2) * W), int((yc + bh / 2) * H)
            else:
                x1, y1, x2, y2 = 0, 0, W, H
            x1, y1 = max(0, x1 - ROI_PAD_PX), max(0, y1 - ROI_PAD_PX)
            x2, y2 = min(W, x2 + ROI_PAD_PX), min(H, y2 + ROI_PAD_PX)
            if x2 <= x1 or y2 <= y1:
                x1, y1, x2, y2 = 0, 0, W, H
            crop = im[y1:y2, x1:x2]
            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            enh = clahe_obj.apply(gray)
            dst_img = roi_root / "images" / split / img_p.name
            cv2.imwrite(str(dst_img), cv2.merge([enh] * 3))
            new_lines = []
            struct_txt = src_lbl_dir / f"{stem}.txt"
            if struct_txt.exists():
                for line in struct_txt.read_text().splitlines():
                    parts = line.split()
                    if len(parts) < 5:
                        continue
                    try:
                        c, sx, sy, sw, sh = map(float, parts[:5])
                    except Exception:
                        continue
                    ax, ay, aw, ah = sx * W, sy * H, sw * W, sh * H
                    nx, ny = (ax - x1) / (x2 - x1), (ay - y1) / (y2 - y1)
                    nw, nh = aw / (x2 - x1), ah / (y2 - y1)
                    if 0 <= nx <= 1 and 0 <= ny <= 1:
                        new_lines.append(f"{int(c)} {nx:.6f} {ny:.6f} {nw:.6f} {nh:.6f}")
            dst_lbl = roi_root / "labels" / split / f"{stem}.txt"
            dst_lbl.write_text("\n".join(new_lines) + ("\n" if new_lines else ""))
            manifest.append({"split": split, "filename": img_p.name, "crop_x1": x1, "crop_y1": y1, "crop_x2": x2, "crop_y2": y2, "label_count": len(new_lines)})
    write_yolo_yaml(roi_root, {0: "CSP", 1: "LV"}, primary_name="data.yaml")
    man = pd.DataFrame(manifest)
    man.to_csv(OUTPUT_ROOT / "tables" / "repaired_roi_enhanced_gt_manifest.csv", index=False)
    ok, counts = dataset_has_expected_images(roi_root)
    log(f"dataset_roi_enhanced_gt repair complete | ok={ok} | counts={counts}")
    return roi_root


def create_data_prepared_alias(dataset_name: str, root: Path):
    if not CREATE_DATA_PREP_ALIASES:
        return None
    root = Path(root)
    alias = DATA_PREP_DIR / dataset_name
    if not root.exists():
        return None
    if alias.resolve() == root.resolve():
        return alias
    if safe_remove_if_empty_or_incomplete(alias):
        pass
    if alias.exists():
        return alias
    alias.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(root, alias, target_is_directory=True)
        mode = "symlink"
    except Exception:
        shutil.copytree(root, alias, symlinks=False)
        mode = "copytree"
    log(f"created data/prepared alias | {dataset_name}: {alias} -> {root} ({mode})")
    return alias


def repair_restored_training_datasets():
    if not REPAIR_RESTORED_WORKSPACE_DATASETS:
        return {"status": "disabled"}
    report = {"started_utc": utc_now()}
    struct_root, struct_total = find_existing_dataset_root("dataset_structures")
    brain_root, brain_total = find_existing_dataset_root("dataset_brain")
    roi_root, roi_total = find_existing_dataset_root("dataset_roi_enhanced_gt")
    report["initial"] = {
        "dataset_structures": {"root": str(struct_root), "total_images": struct_total, "counts": count_images_under_dataset(struct_root)},
        "dataset_brain": {"root": str(brain_root), "total_images": brain_total, "counts": count_images_under_dataset(brain_root)},
        "dataset_roi_enhanced_gt": {"root": str(roi_root), "total_images": roi_total, "counts": count_images_under_dataset(roi_root)},
    }
    ledger = find_source_ledger_materialised()
    report["source_ledger_materialised"] = str(ledger) if ledger else None
    log(f"post-restore dataset status before repair: {json.dumps(report['initial'], indent=2)}")

    # Use root-level DockerRoot directories as canonical training roots.
    canonical_struct = WORKSPACE / "dataset_structures"
    canonical_brain = WORKSPACE / "dataset_brain"
    canonical_roi = WORKSPACE / "dataset_roi_enhanced_gt"

    # If structures exists only under data/prepared, copy/symlink it to root.
    struct_ok, struct_counts = dataset_has_expected_images(struct_root)
    if struct_root and struct_root.exists() and struct_root != canonical_struct and struct_ok and not canonical_struct.exists():
        log(f"promoting dataset_structures to DockerRoot root: {canonical_struct}")
        shutil.copytree(struct_root, canonical_struct, symlinks=False)
        struct_root = canonical_struct
    elif canonical_struct.exists():
        struct_root = canonical_struct

    brain_ok, _ = dataset_has_expected_images(canonical_brain)
    if (not brain_ok) and ledger is not None:
        rebuild_dataset_brain_from_structures_and_ledger(struct_root, canonical_brain, ledger)

    roi_ok, _ = dataset_has_expected_images(canonical_roi)
    brain_ok, _ = dataset_has_expected_images(canonical_brain)
    if (not roi_ok) and brain_ok:
        rebuild_roi_enhanced_from_structures_and_brain(struct_root, canonical_brain, canonical_roi)

    # Always write YAMLs and aliases after repair.
    if canonical_struct.exists():
        write_yolo_yaml(canonical_struct, {0: "CSP", 1: "LV"}, primary_name="data.yaml")
        create_data_prepared_alias("dataset_structures", canonical_struct)
    if canonical_brain.exists():
        write_yolo_yaml(canonical_brain, {0: "Brain"}, primary_name="data_brain.yaml", extra_names=["data.yaml"])
        create_data_prepared_alias("dataset_brain", canonical_brain)
    if canonical_roi.exists():
        write_yolo_yaml(canonical_roi, {0: "CSP", 1: "LV"}, primary_name="data.yaml")
        create_data_prepared_alias("dataset_roi_enhanced_gt", canonical_roi)

    final = {}
    for name in ["dataset_brain", "dataset_structures", "dataset_roi_enhanced_gt"]:
        root = WORKSPACE / name
        ok, counts = dataset_has_expected_images(root)
        final[name] = {"root": str(root), "ok": ok, "counts": counts}
        if not ok:
            record_issue("restore_repair", "warning", f"{name} still incomplete after repair", root=str(root), counts=counts)
    report["final"] = final
    report["completed_utc"] = utc_now()
    save_json(OUTPUT_ROOT / "manifests" / "post_restore_dataset_repair_manifest.json", report)
    print("\nPOST-RESTORE DATASET REPAIR SUMMARY")
    display(pd.DataFrame([{"dataset": k, **v, **{f"{sp}_images": v["counts"].get(sp, 0) for sp in ["train", "val", "test"]}} for k, v in final.items()]))
    return report

archive = choose_workspace_archive_direct()
RESTORE_OK = restore_exported_workspace(archive)
POST_RESTORE_DATASET_REPAIR = repair_restored_training_datasets()
RESTORE_OK


[atlas-train] mounting Google Drive at /content/drive
Mounted at /content/drive
[atlas-train] Google Drive mount status: ok

Manual archive path resolution


,index,candidate_path,exists,size_gb,size
0,0,/content/drive/MyDrive/ATLAS_FN_exports/ATLAS_...,False,NaN,None
1,1,/content/drive/MyDrive/ATLAS_FN_exports/ATLAS_...,True,1.1016,1.10 GB
2,2,/content/drive/MyDrive/ATLAS_FN_exports/ATLAS_...,False,NaN,None
3,3,/content/drive/MyDrive/ATLAS_FN_exports/ATLAS_...,False,NaN,None


[atlas-train] manual archive resolution table written: /content/atlas_fn_workspace/outputs/training_checkpoint_production/tables/manual_archive_path_resolution.csv
[atlas-train] selected explicit workspace archive: /content/drive/MyDrive/ATLAS_FN_exports/ATLAS_FN_dataprep_prepared_plus_audit_20260705_091610.tar.gz (1.10 GB)
[atlas-train] staging archive from Google Drive/local source to local Colab storage
[atlas-train] source=/content/drive/MyDrive/ATLAS_FN_exports/ATLAS_FN_dataprep_prepared_plus_audit_20260705_091610.tar.gz
[atlas-train] target=/content/atlas_fn_archive_cache/ATLAS_FN_dataprep_prepared_plus_audit_20260705_091610.tar.gz
[atlas-train] size=1.10 GB | chunk=32 MB


stage data-prep archive:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

[atlas-train] archive staged | elapsed=22.4s | speed=50.4 MB/s
[atlas-train] verifying staged archive SHA256


...

repair ROI val:   0%|          | 0/575 [00:00<?, ?img/s]

repair ROI test:   0%|          | 0/586 [00:00<?, ?img/s]

[atlas-train] dataset_roi_enhanced_gt repair complete | ok=True | counts={'train': 2629, 'val': 575, 'test': 586}
[atlas-train] created data/prepared alias | dataset_structures: /content/atlas_fn_workspace/data/prepared/dataset_structures -> /content/atlas_fn_workspace/dataset_structures (symlink)
[atlas-train] created data/prepared alias | dataset_brain: /content/atlas_fn_workspace/data/prepared/dataset_brain -> /content/atlas_fn_workspace/dataset_brain (symlink)
[atlas-train] created data/prepared alias | dataset_roi_enhanced_gt: /content/atlas_fn_workspace/data/prepared/dataset_roi_enhanced_gt -> /content/atlas_fn_workspace/dataset_roi_enhanced_gt (symlink)

POST-RESTORE DATASET REPAIR SUMMARY


,dataset,root,ok,counts,train_images,val_images,test_images
0,dataset_brain,/content/atlas_fn_workspace/dataset_brain,True,"{'train': 2629, 'val': 575, 'test': 586}",2629,575,586
1,dataset_structures,/content/atlas_fn_workspace/dataset_structures,True,"{'train': 2629, 'val': 575, 'test': 586}",2629,575,586
2,dataset_roi_enhanced_gt,/content/atlas_fn_workspace/dataset_roi_enhanc...,True,"{'train': 2629, 'val': 575, 'test': 586}",2629,575,586


True

In [ ]:
# ============================================================
# 3. DATASET INTEGRITY PRECHECK — prepared workspace only
# ============================================================
# The final data-prep notebook already established the source-level integrity.
# This cell verifies that the exact prepared datasets needed for training are present.

DATASET_SPECS = {
    "dataset_brain": {
        "root_candidates": [WORKSPACE / "dataset_brain", WORKSPACE / "data/prepared/dataset_brain"],
        "yaml_names": ["data_brain.yaml", "data.yaml"],
        "names": {0: "Brain"},
        "expected_frames": {"train": 2629, "val": 575, "test": 586},
    },
    "dataset_structures": {
        "root_candidates": [WORKSPACE / "dataset_structures", WORKSPACE / "data/prepared/dataset_structures"],
        "yaml_names": ["data.yaml"],
        "names": {0: "CSP", 1: "LV"},
        "expected_frames": {"train": 2629, "val": 575, "test": 586},
    },
    "dataset_roi_enhanced_gt": {
        "root_candidates": [WORKSPACE / "dataset_roi_enhanced_gt", WORKSPACE / "data/prepared/dataset_roi_enhanced_gt"],
        "yaml_names": ["data.yaml"],
        "names": {0: "CSP", 1: "LV"},
        "expected_frames": {"train": 2629, "val": 575, "test": 586},
    },
}

RESOLVED_DATASETS = {}

# If the restore archive was produced by an earlier exporter that omitted root-level
# datasets, repair again here before resolving dataset roots. This is idempotent.
try:
    POST_RESTORE_DATASET_REPAIR
except NameError:
    POST_RESTORE_DATASET_REPAIR = repair_restored_training_datasets()


def count_label_classes(label_dir: Path):
    counts = defaultdict(int)
    nonempty = 0
    files = list(label_dir.glob("*.txt")) if label_dir.exists() else []
    for txt in files:
        text = txt.read_text().strip()
        if text:
            nonempty += 1
        for line in text.splitlines():
            parts = line.split()
            if parts:
                try:
                    counts[int(float(parts[0]))] += 1
                except Exception:
                    pass
    return dict(counts), nonempty, len(files)

def resolve_dataset_root(spec):
    for root in spec["root_candidates"]:
        if root.exists():
            return root
    return spec["root_candidates"][0]

def ensure_yaml(root: Path, names: dict, yaml_names):
    for yn in yaml_names:
        yp = root / yn
        if yp.exists():
            return yp
    # Recreate if folder structure exists.
    yp = root / yaml_names[0]
    yaml_data = {"path": str(root), "train": "images/train", "val": "images/val", "test": "images/test", "names": names}
    yp.parent.mkdir(parents=True, exist_ok=True)
    yp.write_text(yaml.safe_dump(yaml_data, sort_keys=False))
    record_issue("dataset", "warning", "data yaml recreated", root=str(root), yaml=str(yp))
    return yp

inventory = []
for ds, spec in DATASET_SPECS.items():
    root = resolve_dataset_root(spec)
    yaml_path = ensure_yaml(root, spec["names"], spec["yaml_names"])
    RESOLVED_DATASETS[ds] = {"root": root, "yaml": yaml_path}
    for split, expected in spec["expected_frames"].items():
        img_dir = root / "images" / split
        lbl_dir = root / "labels" / split
        imgs = sorted([p for p in img_dir.glob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}]) if img_dir.exists() else []
        labels = sorted(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []
        cls_counts, nonempty, label_files = count_label_classes(lbl_dir)
        ok_parity = len(imgs) == len(labels)
        ok_expected = len(imgs) == expected
        inventory.append({
            "dataset": ds, "split": split, "root": str(root), "yaml": str(yaml_path),
            "images": len(imgs), "labels": len(labels), "nonempty_labels": nonempty,
            "class_0": cls_counts.get(0, 0), "class_1": cls_counts.get(1, 0),
            "expected_images": expected, "parity_ok": ok_parity, "expected_ok": ok_expected,
        })
        status = "PASS" if ok_parity and ok_expected else "FLAG"
        print(f"[{ds}] {split}: images={len(imgs)} labels={len(labels)} expected={expected} parity={ok_parity} => {status}")

DATASET_INVENTORY = pd.DataFrame(inventory)
DATASET_INVENTORY.to_csv(OUTPUT_ROOT / "tables/pre_training_dataset_inventory.csv", index=False)
display(DATASET_INVENTORY)

if DATASET_INVENTORY.empty or not DATASET_INVENTORY["parity_ok"].all() or not DATASET_INVENTORY["expected_ok"].all():
    record_issue("dataset", "warning", "Prepared dataset check has flags after repair; affected training stages may be skipped",
                 parity_ok=bool(DATASET_INVENTORY["parity_ok"].all()) if not DATASET_INVENTORY.empty else False,
                 expected_ok=bool(DATASET_INVENTORY["expected_ok"].all()) if not DATASET_INVENTORY.empty else False)
    maybe_raise("Prepared dataset check failed. Training stages may be skipped.")

BRAIN_YAML = RESOLVED_DATASETS["dataset_brain"]["yaml"]
STRUCT_YAML = RESOLVED_DATASETS["dataset_structures"]["yaml"]
ROI_YAML = RESOLVED_DATASETS["dataset_roi_enhanced_gt"]["yaml"]
BRAIN_DIR = RESOLVED_DATASETS["dataset_brain"]["root"]
STRUCT_DIR = RESOLVED_DATASETS["dataset_structures"]["root"]
ROI_DIR = RESOLVED_DATASETS["dataset_roi_enhanced_gt"]["root"]


[dataset_brain] train: images=2629 labels=2629 expected=2629 parity=True => PASS
[dataset_brain] val: images=575 labels=575 expected=575 parity=True => PASS
[dataset_brain] test: images=586 labels=586 expected=586 parity=True => PASS
[dataset_structures] train: images=2629 labels=2629 expected=2629 parity=True => PASS
[dataset_structures] val: images=575 labels=575 expected=575 parity=True => PASS
[dataset_structures] test: images=586 labels=586 expected=586 parity=True => PASS
[dataset_roi_enhanced_gt] train: images=2629 labels=2629 expected=2629 parity=True => PASS
[dataset_roi_enhanced_gt] val: images=575 labels=575 expected=575 parity=True => PASS
[dataset_roi_enhanced_gt] test: images=586 labels=586 expected=586 parity=True => PASS


,dataset,split,root,yaml,images,labels,nonempty_labels,class_0,class_1,expected_images,parity_ok,expected_ok
0,dataset_brain,train,/content/atlas_fn_workspace/dataset_brain,/content/atlas_fn_workspace/dataset_brain/data...,2629,2629,2629,2629,0,2629,True,True
1,dataset_brain,val,/content/atlas_fn_workspace/dataset_brain,/content/atlas_fn_workspace/dataset_brain/data...,575,575,575,575,0,575,True,True
2,dataset_brain,test,/content/atlas_fn_workspace/dataset_brain,/content/atlas_fn_workspace/dataset_brain/data...,586,586,586,586,0,586,True,True
3,dataset_structures,train,/content/atlas_fn_workspace/dataset_structures,/content/atlas_fn_workspace/dataset_structures...,2629,2629,1566,1258,1040,2629,True,True
4,dataset_structures,val,/content/atlas_fn_workspace/dataset_structures,/content/atlas_fn_workspace/dataset_structures...,575,575,340,277,219,575,True,True
5,dataset_structures,test,/content/atlas_fn_workspace/dataset_structures,/content/atlas_fn_workspace/dataset_structures...,586,586,354,291,222,586,True,True
6,dataset_roi_enhanced_gt,train,/content/atlas_fn_workspace/dataset_roi_enhanc...,/content/atlas_fn_workspace/dataset_roi_enhanc...,2629,2629,1566,1258,1040,2629,True,True
7,dataset_roi_enhanced_gt,val,/content/atlas_fn_workspace/dataset_roi_enhanc...,/content/atlas_fn_workspace/dataset_roi_enhanc...,575,575,340,277,219,575,True,True
8,dataset_roi_enhanced_gt,test,/content/atlas_fn_workspace/dataset_roi_enhanc...,/content/atlas_fn_workspace/dataset_roi_enhanc...,586,586,354,291,222,586,True,True


In [ ]:
# ============================================================
# 4. YOLO26 COMPATIBILITY SHIM + MODEL ASSET DOWNLOAD
# ============================================================
# The shim is defined at notebook module scope so PyTorch can serialize checkpoints.

import ultralytics.nn.modules.block as yolo_block
import ultralytics.nn.modules as yolo_modules
import ultralytics.nn.tasks as yolo_tasks

OriginalSPPF = yolo_block.SPPF
class SPPF_YOLO26_Compatible(OriginalSPPF):
    def __init__(self, c1, c2, k=5, *args, **kwargs):
        super().__init__(c1, c2, k)

SPPF_YOLO26_Compatible.__module__ = "ultralytics.nn.modules.block"
SPPF_YOLO26_Compatible.__name__ = "SPPF_YOLO26_Compatible"
SPPF_YOLO26_Compatible.__qualname__ = "SPPF_YOLO26_Compatible"

def register_yolo26_sppf_compatibility():
    yolo_block.SPPF_YOLO26_Compatible = SPPF_YOLO26_Compatible
    yolo_block.SPPF = SPPF_YOLO26_Compatible
    setattr(yolo_modules, "SPPF_YOLO26_Compatible", SPPF_YOLO26_Compatible)
    setattr(yolo_modules, "SPPF", SPPF_YOLO26_Compatible)
    setattr(yolo_tasks, "SPPF_YOLO26_Compatible", SPPF_YOLO26_Compatible)
    setattr(yolo_tasks, "SPPF", SPPF_YOLO26_Compatible)
    log("YOLO26 SPPF compatibility shim registered")

register_yolo26_sppf_compatibility()

# Serialization smoke test. This catches the previous local-class checkpoint crash before training.
def sppf_serialization_smoke_test():
    tmp = OUTPUT_ROOT / "logs/sppf_serialization_smoke_test.pt"
    obj = {"layer": SPPF_YOLO26_Compatible(16, 16)}
    torch.save(obj, tmp)
    _ = torch.load(tmp, map_location="cpu", weights_only=False)
    tmp.unlink(missing_ok=True)
    log("YOLO26 SPPF serialization smoke test passed")

sppf_serialization_smoke_test()

# Download exact model assets.
def download_asset(url: str, filename: str):
    dest = BASE_WEIGHTS / filename
    if dest.exists() and dest.stat().st_size > 0:
        log(f"reuse model asset: {filename} ({file_size_mb(dest):.2f} MB)")
        return dest
    log(f"download model asset: {filename}")
    torch.hub.download_url_to_file(url, str(dest))
    return dest

YOLO26_DET = download_asset(YOLO26_DET_URL, "yolo26n.pt")
YOLO26_SEG = download_asset(YOLO26_SEG_URL, "yolo26n-seg.pt")
SAM2_SMALL = download_asset(SAM2_SMALL_URL, "sam2_s.pt")

asset_manifest = []
for role, p in {"find_yolo26n": YOLO26_DET, "confirm_grandmaster_yolo26n": YOLO26_DET, "confirm_student_yolo26n_seg": YOLO26_SEG, "measure_yolo26n_seg": YOLO26_SEG, "sam2_small": SAM2_SMALL}.items():
    asset_manifest.append({"role": role, "path": str(p), "filename": p.name, "bytes": p.stat().st_size, "sha256": sha256_file(p)})
ASSET_MANIFEST = pd.DataFrame(asset_manifest)
ASSET_MANIFEST.to_csv(OUTPUT_ROOT / "tables/model_asset_manifest.csv", index=False)
display(ASSET_MANIFEST)


[atlas-train] YOLO26 SPPF compatibility shim registered
[atlas-train] YOLO26 SPPF serialization smoke test passed
[atlas-train] download model asset: yolo26n.pt


100%|██████████| 5.29M/5.29M [00:00<00:00, 164MB/s]


[atlas-train] download model asset: yolo26n-seg.pt


100%|██████████| 6.41M/6.41M [00:00<00:00, 203MB/s]


[atlas-train] download model asset: sam2_s.pt


100%|██████████| 88.0M/88.0M [00:00<00:00, 417MB/s]


,role,path,filename,bytes,sha256
0,find_yolo26n,/content/atlas_fn_workspace/base_weights/yolo2...,yolo26n.pt,5544453,9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0...
1,confirm_grandmaster_yolo26n,/content/atlas_fn_workspace/base_weights/yolo2...,yolo26n.pt,5544453,9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0...
2,confirm_student_yolo26n_seg,/content/atlas_fn_workspace/base_weights/yolo2...,yolo26n-seg.pt,6719965,361fbfabab285c3237700b6bb91d7ecfa602cd945fffda...
3,measure_yolo26n_seg,/content/atlas_fn_workspace/base_weights/yolo2...,yolo26n-seg.pt,6719965,361fbfabab285c3237700b6bb91d7ecfa602cd945fffda...
4,sam2_small,/content/atlas_fn_workspace/base_weights/sam2_...,sam2_s.pt,92278178,6ba91d739a6adfc1dcaf76336b2f380f3eae8d24a88f56...


In [ ]:
# ============================================================
# 5. TRAINING UTILITIES — resumable, auditable, DockerRoot semantics
# ============================================================

def best_or_last(run_dir: Path):
    best = run_dir / "weights" / "best.pt"
    last = run_dir / "weights" / "last.pt"
    if best.exists():
        return best, "best"
    if REUSE_LAST_PT_IF_BEST_MISSING and last.exists():
        record_issue("checkpoint", "warning", "using last.pt because best.pt is unavailable", run=str(run_dir))
        return last, "last"
    return best, "missing"

def yolo_cache_arg():
    if YOLO_CACHE_MODE in {"ram", "disk"}:
        return YOLO_CACHE_MODE
    return False

def validate_yaml_has_data(yaml_path: Path, require_train_val=True):
    yaml_path = Path(yaml_path)
    if not yaml_path.exists():
        return False, f"missing yaml: {yaml_path}"
    data = yaml.safe_load(yaml_path.read_text())
    root = Path(data.get("path", yaml_path.parent))
    def count_items(entry):
        if entry is None: return 0
        p = Path(entry)
        if not p.is_absolute(): p = root / entry
        if p.is_file() and p.suffix == ".txt":
            return len([x for x in p.read_text().splitlines() if x.strip()])
        if p.exists() and p.is_dir():
            return len([x for x in p.glob("*") if x.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}])
        return 0
    train_n = count_items(data.get("train"))
    val_n = count_items(data.get("val"))
    if require_train_val and (train_n == 0 or val_n == 0):
        return False, f"empty train/val in {yaml_path}: train={train_n}, val={val_n}"
    return True, f"train={train_n}, val={val_n}"

def run_yolo_train(stage_name: str, base_model: Path, data_yaml: Path, project: Path, name: str, task: str = None, **kwargs):
    register_yolo26_sppf_compatibility()
    run_dir = project / name
    existing, existing_kind = best_or_last(run_dir)
    if existing.exists() and RESUME_EXISTING_ARTIFACTS and not FORCE_RETRAIN:
        record_stage(stage_name, "reused", checkpoint=str(existing), checkpoint_kind=existing_kind)
        return str(existing)

    ok, msg = validate_yaml_has_data(data_yaml)
    if not ok:
        record_stage(stage_name, "skipped", reason=msg, data_yaml=str(data_yaml))
        if STOP_ON_CRITICAL: raise RuntimeError(msg)
        return None

    if FORCE_RETRAIN and run_dir.exists():
        shutil.rmtree(run_dir)

    log(f"start training {stage_name} | base={base_model.name}, data={data_yaml}, name={name}")
    clean_system()
    model = YOLO(str(base_model)) if task is None else YOLO(str(base_model), task=task)
    train_args = dict(data=str(data_yaml), project=str(project), name=name, exist_ok=True, device=0 if torch.cuda.is_available() else "cpu")
    train_args.update(kwargs)
    # Every YOLO call receives the manuscript/reproduction seed unless a candidate
    # deliberately overrides it and records that override in the ablation contract.
    train_args.setdefault("seed", SEED)
    if "cache" not in train_args:
        train_args["cache"] = yolo_cache_arg()

    # Persist exact training arguments before training starts.
    args_path = OUTPUT_ROOT / "manifests" / f"train_args_{stage_name}.json"
    save_json(args_path, {k: str(v) if isinstance(v, Path) else v for k, v in train_args.items()})

    try:
        model.train(**train_args)
    except Exception as e:
        record_stage(stage_name, "failed", error=repr(e), trace=traceback.format_exc()[-4000:])
        if STOP_ON_CRITICAL: raise
        return None
    finally:
        del model
        clean_system()

    ckpt, kind = best_or_last(run_dir)
    if ckpt.exists():
        record_stage(stage_name, "trained", checkpoint=str(ckpt), checkpoint_kind=kind, bytes=ckpt.stat().st_size)
        return str(ckpt)
    record_stage(stage_name, "missing_checkpoint_after_train", run_dir=str(run_dir))
    return None

def checkpoint_row(role, ckpt_path, source_stage=None):
    if not ckpt_path:
        return {"role": role, "status": "missing", "path": None}
    p = Path(ckpt_path)
    return {
        "role": role,
        "status": "exists" if p.exists() else "missing",
        "path": str(p),
        "filename": p.name if p.exists() else None,
        "bytes": p.stat().st_size if p.exists() else None,
        "sha256": sha256_file(p) if p.exists() else None,
        "source_stage": source_stage,
    }


In [ ]:

# ============================================================
# 5B. BMC TRAINING CONTRACT + OPTIMISED FIND ABLATION PLAN
# ============================================================
# BMC_LOCKED is the manuscript reproduction checkpoint. OPTIMISED_* models are
# new research artefacts and are always exported/labelled separately.

RUN_FIND_HYPERPARAMETER_ABLATION = True
FORCE_RETRAIN_FIND_ABLATION = False       # set True for a fresh optimisation run

# Defensive seed guard for partial reruns. If this cell is run independently,
# restore the BMC manuscript seed rather than failing with NameError.
try:
    SEED
except NameError:
    GLOBAL_SEED = 42
    SEED = 42

OPTIMISED_SELECTION_METRIC_PRIORITY = [
    'metrics/mAP50(B)',
    'metrics/mAP50-95(B)',
    'fitness',
]

BMC_FIND_TRAINING_CONTRACT = {
    'model_label': 'BMC_LOCKED',
    'role': 'manuscript_reproduction',
    'run_name': 'BrainLocator_YOLO26n_MuSGD',
    'imgsz': 640,
    'epochs': 50,
    'patience': 10,
    'batch': 16,
    'optimizer': 'auto',
    'lr0': 0.001,
    'lrf': 0.01,
    'weight_decay': 0.0005,
    'cos_lr': True,
    'mosaic': 0.5,
    'fliplr': 0.5,
    'flipud': 0.5,
    'scale': 0.5,
    'seed': SEED,
    'threshold_for_evidence': 0.25,
    'selection_rule': 'best validation-loss checkpoint; no locked-test tuning',
}

# A compact but substantive Find ablation grid. Each candidate changes one or two
# clinically relevant training assumptions while retaining the same patient-exclusive data.
FIND_OPTIMISATION_GRID = [
    {
        'model_label': 'OPT_A_low_mosaic_longer_patience',
        'role': 'optimised_candidate',
        'run_name': 'BrainLocator_YOLO26n_OPT_A_low_mosaic_longer_patience',
        'rationale': 'lower mosaic and longer patience to reduce box-scale distortion while allowing convergence',
        'imgsz': 640, 'epochs': 75, 'patience': 15, 'batch': 16,
        'optimizer': 'auto', 'lr0': 0.001, 'lrf': 0.01, 'weight_decay': 0.0005,
        'cos_lr': True, 'mosaic': 0.20, 'fliplr': 0.5, 'flipud': 0.5, 'scale': 0.35,
    },
    {
        'model_label': 'OPT_B_no_mosaic_lowlr',
        'role': 'optimised_candidate',
        'run_name': 'BrainLocator_YOLO26n_OPT_B_no_mosaic_lowlr',
        'rationale': 'remove mosaic and use lower lr to preserve fetal-head geometry for deterministic Measure',
        'imgsz': 640, 'epochs': 75, 'patience': 15, 'batch': 16,
        'optimizer': 'auto', 'lr0': 0.0008, 'lrf': 0.01, 'weight_decay': 0.0005,
        'cos_lr': True, 'mosaic': 0.00, 'fliplr': 0.5, 'flipud': 0.0, 'scale': 0.25,
    },
    {
        'model_label': 'OPT_C_higher_resolution',
        'role': 'optimised_candidate',
        'run_name': 'BrainLocator_YOLO26n_OPT_C_higher_resolution',
        'rationale': 'higher input size to test whether localization/ellipse geometry improves at modest extra cost',
        'imgsz': 800, 'epochs': 75, 'patience': 15, 'batch': 8,
        'optimizer': 'auto', 'lr0': 0.001, 'lrf': 0.01, 'weight_decay': 0.0005,
        'cos_lr': True, 'mosaic': 0.20, 'fliplr': 0.5, 'flipud': 0.5, 'scale': 0.35,
    },
]

for _cfg in FIND_OPTIMISATION_GRID:
    _cfg.setdefault('seed', SEED)

contract_df = pd.DataFrame([BMC_FIND_TRAINING_CONTRACT] + FIND_OPTIMISATION_GRID)
(OUTPUT_ROOT / 'tables').mkdir(parents=True, exist_ok=True)
contract_df.to_csv(OUTPUT_ROOT / 'tables/find_bmc_vs_optimised_training_contract.csv', index=False)
print('Find training contract and optimisation grid')
display(contract_df[['model_label','role','run_name','imgsz','epochs','patience','batch','lr0','mosaic','fliplr','flipud','scale','seed','rationale'] if 'rationale' in contract_df.columns else contract_df.columns])


Find training contract and optimisation grid


,model_label,role,run_name,imgsz,epochs,patience,batch,lr0,mosaic,fliplr,flipud,scale,seed,rationale
0,BMC_LOCKED,manuscript_reproduction,BrainLocator_YOLO26n_MuSGD,640,50,10,16,0.0010,0.5,0.5,0.5,0.50,42,NaN
1,OPT_A_low_mosaic_longer_patience,optimised_candidate,BrainLocator_YOLO26n_OPT_A_low_mosaic_longer_p...,640,75,15,16,0.0010,0.2,0.5,0.5,0.35,42,lower mosaic and longer patience to reduce box...
2,OPT_B_no_mosaic_lowlr,optimised_candidate,BrainLocator_YOLO26n_OPT_B_no_mosaic_lowlr,640,75,15,16,0.0008,0.0,0.5,0.0,0.25,42,remove mosaic and use lower lr to preserve fet...
3,OPT_C_higher_resolution,optimised_candidate,BrainLocator_YOLO26n_OPT_C_higher_resolution,800,75,15,8,0.0010,0.2,0.5,0.5,0.35,42,higher input size to test whether localization...


In [ ]:
# ============================================================
# 5C. SEED CONTRACT AUDIT — BMC and optimised checkpoints
# ============================================================
# Confirms the global seed is defined before any training stage and records it as a
# first-class reproducibility field for downstream manuscript/package audits.

try:
    SEED
except NameError as e:
    raise RuntimeError("SEED must be defined before checkpoint training. Run cells 0–1 from the top.") from e

seed_contract_rows = [
    {
        "field": "GLOBAL_SEED",
        "value": int(GLOBAL_SEED),
        "expected_bmc_value": 42,
        "status": "PASS" if int(GLOBAL_SEED) == 42 else "FLAG",
    },
    {
        "field": "SEED",
        "value": int(SEED),
        "expected_bmc_value": 42,
        "status": "PASS" if int(SEED) == 42 else "FLAG",
    },
    {
        "field": "BMC_FIND_TRAINING_CONTRACT.seed",
        "value": int(BMC_FIND_TRAINING_CONTRACT.get("seed")),
        "expected_bmc_value": 42,
        "status": "PASS" if int(BMC_FIND_TRAINING_CONTRACT.get("seed")) == 42 else "FLAG",
    },
]
for cfg in FIND_OPTIMISATION_GRID:
    seed_contract_rows.append({
        "field": f"{cfg.get('model_label')}.seed",
        "value": int(cfg.get("seed", SEED)),
        "expected_bmc_value": 42,
        "status": "PASS" if int(cfg.get("seed", SEED)) == 42 else "FLAG",
    })

seed_contract_audit = pd.DataFrame(seed_contract_rows)
seed_contract_audit.to_csv(OUTPUT_ROOT / "tables/checkpoint_training_seed_contract_audit.csv", index=False)
display(seed_contract_audit)
if not (seed_contract_audit["status"] == "PASS").all():
    raise RuntimeError("Seed contract audit failed; BMC-locked and optimised training must use declared seed values.")


,field,value,expected_bmc_value,status
0,GLOBAL_SEED,42,42,PASS
1,SEED,42,42,PASS
2,BMC_FIND_TRAINING_CONTRACT.seed,42,42,PASS
3,OPT_A_low_mosaic_longer_patience.seed,42,42,PASS
4,OPT_B_no_mosaic_lowlr.seed,42,42,PASS
5,OPT_C_higher_resolution.seed,42,42,PASS


In [ ]:
# ============================================================
# 6. DOCKERROOT CELL 4A REPLICA — Find / BrainLocator YOLO26n training
# ============================================================
FIND_RUN_NAME = "BrainLocator_YOLO26n_MuSGD"
FIND_CKPT = None

if RUN_FIND_TRAINING:
    FIND_CKPT = run_yolo_train(
        stage_name="find_brainlocator_yolo26n",
        base_model=YOLO26_DET,
        data_yaml=BRAIN_YAML,
        project=RUNS_DIR,
        name=FIND_RUN_NAME,
        imgsz=640,
        epochs=2 if SMOKE_TEST else 50,
        patience=2 if SMOKE_TEST else 10,
        batch=16,
        optimizer="auto",
        warmup_epochs=5,
        cos_lr=True,
        lr0=0.001,
        lrf=0.01,
        weight_decay=0.0005,
        mosaic=0.5,
        fliplr=0.5,
        flipud=0.5,
        scale=0.5,
        nbs=64,
        seed=SEED,
        val=True,
        plots=True,
        workers=0,
        verbose=True,
        amp=True,
    )
else:
    FIND_CKPT, _ = best_or_last(RUNS_DIR / FIND_RUN_NAME)
    FIND_CKPT = str(FIND_CKPT) if Path(FIND_CKPT).exists() else None

FIND_CKPT


[atlas-train] YOLO26 SPPF compatibility shim registered
[atlas-train] start training find_brainlocator_yolo26n | base=yolo26n.pt, data=/content/atlas_fn_workspace/dataset_brain/data_brain.yaml, name=BrainLocator_YOLO26n_MuSGD
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/atlas_fn_workspace/dataset_brain/data_brain.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, 

'/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt'

In [ ]:

# ============================================================
# 6B. FIND HYPERPARAMETER ABLATION + OPTIMISED CHECKPOINT SELECTION
# ============================================================
# Runs after BMC Find training. Produces tables/figures comparing manuscript parameters
# with candidate optimised Find checkpoints. Selection is based only on validation metrics.

from math import isnan


def _read_yolo_results(run_dir: Path) -> dict:
    run_dir = Path(run_dir)
    csv_path = run_dir / 'results.csv'
    out = {'run_dir': str(run_dir), 'results_csv': str(csv_path), 'results_exists': csv_path.exists()}
    if not csv_path.exists():
        return out
    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            return out
        df.columns = [c.strip() for c in df.columns]
        last = df.iloc[-1].to_dict()
        best = {}
        for c in df.columns:
            if c == 'epoch':
                continue
            vals = pd.to_numeric(df[c], errors='coerce')
            if vals.notna().any():
                best[f'best_{c}'] = float(vals.max())
                best[f'final_{c}'] = float(vals.iloc[-1]) if pd.notna(vals.iloc[-1]) else np.nan
        out.update(best)
        out['epochs_completed'] = int(pd.to_numeric(df.get('epoch', pd.Series([len(df)-1])), errors='coerce').max() + 1)
    except Exception as e:
        out['parse_error'] = repr(e)
    return out


def _metric_for_selection(row: dict):
    for key in OPTIMISED_SELECTION_METRIC_PRIORITY:
        for prefix in ['best_', 'final_', '']:
            k = prefix + key
            if k in row:
                try:
                    v = float(row[k])
                    if np.isfinite(v):
                        return v, k
                except Exception:
                    pass
    return np.nan, None


def _train_find_candidate(cfg: dict) -> dict:
    name = cfg['run_name']
    run_dir = RUNS_DIR / name
    if FORCE_RETRAIN_FIND_ABLATION and run_dir.exists():
        shutil.rmtree(run_dir)
    args = {k: v for k, v in cfg.items() if k in {
        'imgsz','epochs','patience','batch','optimizer','lr0','lrf','weight_decay','cos_lr','mosaic','fliplr','flipud','scale'
    }}
    # Reproducibility/safety controls.
    args.update(dict(seed=SEED, val=True, plots=True, workers=0, verbose=True, amp=False))
    if SMOKE_TEST:
        args['epochs'] = min(2, int(args.get('epochs', 2)))
        args['patience'] = min(2, int(args.get('patience', 2)))
    ckpt = run_yolo_train(
        stage_name=f"find_ablation_{cfg['model_label']}",
        base_model=YOLO26_DET,
        data_yaml=BRAIN_YAML,
        project=RUNS_DIR,
        name=name,
        **args,
    )
    row = dict(cfg)
    row['checkpoint'] = ckpt
    row['checkpoint_exists'] = bool(ckpt and Path(ckpt).exists())
    row['checkpoint_sha256'] = sha256_file(Path(ckpt)) if row['checkpoint_exists'] else None
    row.update(_read_yolo_results(run_dir))
    score, score_key = _metric_for_selection(row)
    row['selection_score'] = score
    row['selection_score_key'] = score_key
    return row

# BMC row from already-trained/reused FIND_CKPT.
bmc_row = dict(BMC_FIND_TRAINING_CONTRACT)
bmc_row['checkpoint'] = FIND_CKPT
bmc_row['checkpoint_exists'] = bool(FIND_CKPT and Path(FIND_CKPT).exists())
bmc_row['checkpoint_sha256'] = sha256_file(Path(FIND_CKPT)) if bmc_row['checkpoint_exists'] else None
bmc_row.update(_read_yolo_results(RUNS_DIR / BMC_FIND_TRAINING_CONTRACT['run_name']))
score, score_key = _metric_for_selection(bmc_row)
bmc_row['selection_score'] = score
bmc_row['selection_score_key'] = score_key

ablation_rows = [bmc_row]
if RUN_FIND_HYPERPARAMETER_ABLATION:
    for cfg in FIND_OPTIMISATION_GRID:
        ablation_rows.append(_train_find_candidate(cfg))
else:
    for cfg in FIND_OPTIMISATION_GRID:
        row = dict(cfg)
        row.update({'checkpoint': None, 'checkpoint_exists': False, 'status': 'not_run'})
        ablation_rows.append(row)

FIND_HPARAM_RESULTS = pd.DataFrame(ablation_rows)
FIND_HPARAM_RESULTS.to_csv(OUTPUT_ROOT / 'tables/find_hyperparameter_tuning_results.csv', index=False)

# Select optimised from candidates only; BMC remains the manuscript checkpoint.
cand = FIND_HPARAM_RESULTS[(FIND_HPARAM_RESULTS['role'] == 'optimised_candidate') & (FIND_HPARAM_RESULTS['checkpoint_exists'] == True)].copy()
if not cand.empty and 'selection_score' in cand.columns:
    cand = cand.sort_values('selection_score', ascending=False, na_position='last')
    best = cand.iloc[0]
    OPTIMIZED_FIND_CKPT = str(best['checkpoint'])
    OPTIMIZED_FIND_RUN_NAME = str(best['run_name'])
    OPTIMIZED_FIND_MODEL_LABEL = str(best['model_label'])
    OPTIMIZED_FIND_SELECTION_SCORE = float(best['selection_score']) if pd.notna(best['selection_score']) else None
    # Stable export copy with an unambiguous optimized name.
    opt_dir = RUNS_DIR / 'BrainLocator_YOLO26n_OPTIMISED' / 'weights'
    opt_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(OPTIMIZED_FIND_CKPT, opt_dir / 'best.pt')
    OPTIMIZED_FIND_CKPT = str(opt_dir / 'best.pt')
else:
    OPTIMIZED_FIND_CKPT = None
    OPTIMIZED_FIND_RUN_NAME = None
    OPTIMIZED_FIND_MODEL_LABEL = None
    OPTIMIZED_FIND_SELECTION_SCORE = None

selection_summary = {
    'bmc_checkpoint': str(FIND_CKPT),
    'bmc_model_label': 'BMC_LOCKED',
    'optimised_checkpoint': OPTIMIZED_FIND_CKPT,
    'optimised_source_run': OPTIMIZED_FIND_RUN_NAME,
    'optimised_model_label': OPTIMIZED_FIND_MODEL_LABEL,
    'optimised_selection_score': OPTIMIZED_FIND_SELECTION_SCORE,
    'selection_metric_priority': OPTIMISED_SELECTION_METRIC_PRIORITY,
    'selection_boundary': 'optimised checkpoint is a new research artefact; manuscript tables retain BMC_LOCKED checkpoint',
}
save_json(OUTPUT_ROOT / 'manifests/find_optimised_selection_summary.json', selection_summary)

# Figures for reviewer-facing audit.
(OUTPUT_ROOT / 'figures').mkdir(parents=True, exist_ok=True)
try:
    import matplotlib.pyplot as plt
    plot_df = FIND_HPARAM_RESULTS.copy()
    plot_df['plot_score'] = pd.to_numeric(plot_df.get('selection_score'), errors='coerce')
    fig, ax = plt.subplots(figsize=(10, 4.8))
    labels = plot_df['model_label'].astype(str).tolist()
    ax.bar(range(len(plot_df)), plot_df['plot_score'].fillna(0).values)
    ax.set_xticks(range(len(plot_df)))
    ax.set_xticklabels(labels, rotation=35, ha='right')
    ax.set_ylabel('Validation selection score')
    ax.set_title('Find hyperparameter ablation: BMC-locked vs optimised candidates')
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / 'figures/find_hyperparameter_bmc_vs_optimised_score.png', dpi=180)
    plt.close(fig)
except Exception as e:
    record_issue('hparam_figures', 'warning', 'could not generate hyperparameter figure', error=repr(e))

print('Find hyperparameter / ablation results')
display(FIND_HPARAM_RESULTS)
print('Find optimised selection summary')
print(json.dumps(selection_summary, indent=2))


[atlas-train] YOLO26 SPPF compatibility shim registered
[atlas-train] start training find_ablation_OPT_A_low_mosaic_longer_patience | base=yolo26n.pt, data=/content/atlas_fn_workspace/dataset_brain/data_brain.yaml, name=BrainLocator_YOLO26n_OPT_A_low_mosaic_longer_patience
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/atlas_fn_workspace/dataset_brain/data_brain.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=75, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=

,model_label,role,run_name,imgsz,epochs,patience,batch,optimizer,lr0,lrf,...,best_lr/pg0,final_lr/pg0,best_lr/pg1,final_lr/pg1,best_lr/pg2,final_lr/pg2,epochs_completed,selection_score,selection_score_key,rationale
0,BMC_LOCKED,manuscript_reproduction,BrainLocator_YOLO26n_MuSGD,640,50,10,16,auto,0.0010,0.01,...,0.001967,0.000022,0.001967,0.000022,0.001967,0.000022,51,0.99500,best_metrics/mAP50(B),NaN
1,OPT_A_low_mosaic_longer_patience,optimised_candidate,BrainLocator_YOLO26n_OPT_A_low_mosaic_longer_p...,640,75,15,16,auto,0.0010,0.01,...,0.001992,0.001786,0.001992,0.001786,0.001992,0.001786,18,0.99497,best_metrics/mAP50(B),lower mosaic and longer patience to reduce box...
2,OPT_B_no_mosaic_lowlr,optimised_candidate,BrainLocator_YOLO26n_OPT_B_no_mosaic_lowlr,640,75,15,16,auto,0.0008,0.01,...,0.001992,0.000042,0.001992,0.000042,0.001992,0.000042,72,0.99500,best_metrics/mAP50(B),remove mosaic and use lower lr to preserve fet...
3,OPT_C_higher_resolution,optimised_candidate,BrainLocator_YOLO26n_OPT_C_higher_resolution,800,75,15,8,auto,0.0010,0.01,...,0.001995,0.000021,0.001995,0.000021,0.001995,0.000021,76,0.99500,best_metrics/mAP50(B),higher input size to test whether localization...


Find optimised selection summary
{
  "bmc_checkpoint": "/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt",
  "bmc_model_label": "BMC_LOCKED",
  "optimised_checkpoint": "/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_OPTIMISED/weights/best.pt",
  "optimised_source_run": "BrainLocator_YOLO26n_OPT_B_no_mosaic_lowlr",
  "optimised_model_label": "OPT_B_no_mosaic_lowlr",
  "optimised_selection_score": 0.995,
  "selection_metric_priority": [
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
    "fitness"
  ],
  "selection_boundary": "optimised checkpoint is a new research artefact; manuscript tables retain BMC_LOCKED checkpoint"
}


In [ ]:
# ============================================================
# 7. DOCKERROOT GRANDMASTER REPLICA — Confirm staged training S0–S5
# ============================================================
# Replicates DockerRoot Cell 4: staged curriculum, S1 weight average and S3 task-vector merge.

RUN_NAME = "UniDet_GrandMaster_YOLO26"

STAGE_HYPS = {
    "S0": {"box": 7.5, "cls": 0.5, "dfl": 3.0, "lr0": 0.001},
    "S1": {"box": 6.0, "cls": 4.0, "dfl": 1.5, "lr0": 0.001},
    "S2": {"box": 7.5, "cls": 0.5, "dfl": 1.5, "lr0": 0.002},
    "S3": {"box": 9.5, "cls": 0.5, "dfl": 2.0, "lr0": 0.001},
    "S4": {"box": 7.5, "cls": 0.5, "dfl": 1.5, "lr0": 0.0005},
    "S5": {"box": 10.0,"cls": 0.5, "dfl": 2.5, "lr0": 0.0001},
}

STAGES = {
    "S0": {"name": "S0_Macro", "epochs": 20, "imgsz": 640, "batch": 32, "targets": [], "use_roi": False, "mosaic": 0.5},
    "S1_CSP": {"name": "S1_GT_CSP", "epochs": 15, "imgsz": 640, "batch": 32, "targets": ["thalamic"], "use_roi": True, "mosaic": 0.0},
    "S1_LV":  {"name": "S1_GT_LV",  "epochs": 15, "imgsz": 640, "batch": 32, "targets": ["ventricular"], "use_roi": True, "mosaic": 0.0},
    "S2": {"name": "S2_Foundation", "epochs": 40, "imgsz": 640, "batch": 32, "targets": [], "use_roi": True, "mosaic": 0.5},
    "S3_CSP": {"name": "S3_Hard_CSP", "epochs": 30, "imgsz": 800, "batch": 16, "targets": ["diverse"], "use_roi": True, "aug": True},
    "S3_LV":  {"name": "S3_Hard_LV",  "epochs": 30, "imgsz": 800, "batch": 16, "targets": ["diverse"], "use_roi": True, "aug": True},
    "S4": {"name": "S4_Fusion", "epochs": 20, "imgsz": 640, "batch": 24, "targets": [], "use_roi": True, "mosaic": 0.2},
    "S5": {"name": "S5_PixelPerfect", "epochs": 15, "imgsz": 800, "batch": 12, "targets": [], "use_roi": True, "mosaic": 0.0},
}

# Locate plane_context.csv from the data-prep notebook.
def resolve_plane_context():
    candidates = [STRUCT_DIR / "plane_context.csv", WORKSPACE / "dataset_structures/plane_context.csv", WORKSPACE / "outputs/tables/plane_context.csv", WORKSPACE / "outputs/plane_context.csv"]
    for p in candidates:
        if p.exists():
            return p
    # Build fallback from source ledger if available.
    ledgers = list(WORKSPACE.rglob("source_ledger_materialised.csv")) + list(WORKSPACE.rglob("source_ledger*.csv"))
    for p in ledgers:
        try:
            df = pd.read_csv(p)
            if {"filename", "split", "plane"}.issubset(df.columns):
                out = STRUCT_DIR / "plane_context.csv"
                df[["filename", "split", "plane"]].drop_duplicates().to_csv(out, index=False)
                record_issue("grandmaster", "warning", "plane_context.csv rebuilt from source ledger", source=str(p), out=str(out))
                return out
        except Exception:
            pass
    record_issue("grandmaster", "error", "plane_context.csv unavailable; stage subset YAMLs will use all images")
    return None

PLANE_CONTEXT = resolve_plane_context()

from copy import deepcopy

def create_stage_yaml(task_key: str, keywords, use_roi: bool):
    subset_dir = RUNS_DIR / "subsets" / task_key
    subset_dir.mkdir(parents=True, exist_ok=True)
    base = ROI_DIR if use_roi else STRUCT_DIR
    yaml_data = {"path": str(base), "names": {0: "CSP", 1: "LV"}}

    if PLANE_CONTEXT and Path(PLANE_CONTEXT).exists():
        df = pd.read_csv(PLANE_CONTEXT)
        if keywords and "plane" in df.columns:
            mask = df["plane"].apply(lambda x: any(k.lower() in str(x).lower() for k in keywords))
            target_df = df[mask]
        else:
            target_df = df
    else:
        target_df = None

    for split in ["train", "val"]:
        img_dir = base / "images" / split
        if target_df is not None and "filename" in target_df.columns and "split" in target_df.columns:
            stems = set(target_df[target_df["split"] == split]["filename"].apply(lambda x: Path(str(x)).stem))
            found = [str(p) for p in img_dir.glob("*") if p.stem in stems]
        else:
            found = [str(p) for p in img_dir.glob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}]
        txt = subset_dir / f"{split}.txt"
        txt.write_text("\n".join(found))
        yaml_data[split] = str(txt)
        log(f"stage {task_key} {split}: {len(found)} images | base={base.name} | keywords={keywords}")

    out = subset_dir / "data.yaml"
    out.write_text(yaml.safe_dump(yaml_data, sort_keys=False))
    return out

def average_checkpoints(ckpt_a, ckpt_b, out_path: Path, label: str):
    if out_path.exists() and RESUME_EXISTING_ARTIFACTS and not FORCE_RETRAIN:
        record_stage(label, "reused", checkpoint=str(out_path))
        return str(out_path)
    if not ckpt_a or not ckpt_b or not Path(ckpt_a).exists() or not Path(ckpt_b).exists():
        record_stage(label, "skipped", reason="source checkpoints missing", ckpt_a=str(ckpt_a), ckpt_b=str(ckpt_b))
        return None
    clean_system()
    ckpt1 = torch.load(ckpt_a, map_location="cpu", weights_only=False)
    ckpt2 = torch.load(ckpt_b, map_location="cpu", weights_only=False)
    m1, m2 = ckpt1["model"].float(), ckpt2["model"].float()
    sd1, sd2 = m1.state_dict(), m2.state_dict()
    new_sd = {}
    for k in sd1:
        if k in sd2 and torch.is_floating_point(sd1[k]):
            new_sd[k] = (sd1[k] + sd2[k]) / 2.0
        else:
            new_sd[k] = sd1[k]
    m1.load_state_dict(new_sd, strict=False)
    ckpt1["model"] = m1
    out_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(ckpt1, out_path)
    record_stage(label, "merged", checkpoint=str(out_path), sha256=sha256_file(out_path))
    del ckpt1, ckpt2, m1, m2
    clean_system()
    return str(out_path)

def task_vector_merge(base_ckpt, ckpt_a, ckpt_b, out_path: Path, label: str, alpha=0.5):
    if out_path.exists() and RESUME_EXISTING_ARTIFACTS and not FORCE_RETRAIN:
        record_stage(label, "reused", checkpoint=str(out_path))
        return str(out_path)
    if not all([base_ckpt, ckpt_a, ckpt_b]) or not all(Path(p).exists() for p in [base_ckpt, ckpt_a, ckpt_b]):
        record_stage(label, "skipped", reason="source checkpoints missing", base=str(base_ckpt), ckpt_a=str(ckpt_a), ckpt_b=str(ckpt_b))
        return None
    clean_system()
    base = torch.load(base_ckpt, map_location="cpu", weights_only=False)
    sda = torch.load(ckpt_a, map_location="cpu", weights_only=False)["model"].float().state_dict()
    sdb = torch.load(ckpt_b, map_location="cpu", weights_only=False)["model"].float().state_dict()
    model = base["model"].float()
    bsd = model.state_dict()
    new_sd = deepcopy(bsd)
    for k in bsd:
        if k in sda and k in sdb and torch.is_floating_point(bsd[k]):
            new_sd[k] = bsd[k] + ((sda[k] - bsd[k]) + (sdb[k] - bsd[k])) * alpha
    model.load_state_dict(new_sd, strict=False)
    base["model"] = model
    out_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(base, out_path)
    record_stage(label, "merged", checkpoint=str(out_path), sha256=sha256_file(out_path))
    del base, model, sda, sdb
    clean_system()
    return str(out_path)

def train_confirm_stage(key: str, resume_w=None):
    cfg = STAGES[key]
    hyps = STAGE_HYPS.get(key[:2], STAGE_HYPS["S2"])
    run_id = f"{RUN_NAME}_{cfg['name']}"
    yaml_path = create_stage_yaml(key, cfg["targets"], cfg["use_roi"])
    base = Path(resume_w) if resume_w else YOLO26_DET
    return run_yolo_train(
        stage_name=f"confirm_grandmaster_{key}",
        base_model=base,
        data_yaml=yaml_path,
        project=RUNS_DIR,
        name=run_id,
        imgsz=cfg["imgsz"],
        epochs=1 if SMOKE_TEST else cfg["epochs"],
        batch=cfg["batch"],
        box=hyps["box"], cls=hyps["cls"], dfl=hyps["dfl"], lr0=hyps["lr0"],
        augment=cfg.get("aug", False),
        mosaic=cfg.get("mosaic", 1.0),
        optimizer="auto", warmup_epochs=5, cos_lr=True, workers=0,
        cache=False if YOLO_CACHE_MODE == "off" else yolo_cache_arg(),
        amp=True, verbose=False, plots=True,
    )

CONFIRM_GRANDMASTER_CKPT = None
if RUN_GRANDMASTER_CONFIRM_TRAINING:
    w0 = train_confirm_stage("S0")
    w1c = train_confirm_stage("S1_CSP", w0)
    w1l = train_confirm_stage("S1_LV", w0)
    fed = average_checkpoints(w1c, w1l, RUNS_DIR / "S1_FEDERATED.pt", "confirm_S1_federated_average")
    w2 = train_confirm_stage("S2", fed)
    w3c = train_confirm_stage("S3_CSP", w2)
    w3l = train_confirm_stage("S3_LV", w2)
    merged = task_vector_merge(w2, w3c, w3l, RUNS_DIR / "S3_MERGED.pt", "confirm_S3_task_vector_merge")
    w4 = train_confirm_stage("S4", merged)
    CONFIRM_GRANDMASTER_CKPT = train_confirm_stage("S5", w4)
else:
    cand, _ = best_or_last(RUNS_DIR / f"{RUN_NAME}_S5_PixelPerfect")
    CONFIRM_GRANDMASTER_CKPT = str(cand) if cand.exists() else None

CONFIRM_GRANDMASTER_CKPT


[atlas-train] stage S0 train: 2629 images | base=dataset_structures | keywords=[]
[atlas-train] stage S0 val: 575 images | base=dataset_structures | keywords=[]
[atlas-train] YOLO26 SPPF compatibility shim registered
[atlas-train] start training confirm_grandmaster_S0 | base=yolo26n.pt, data=/content/atlas_fn_workspace/Training_Runs/subsets/S0/data.yaml, name=UniDet_GrandMaster_YOLO26_S0_Macro
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/atlas_fn_workspace/Training_Runs/subsets/S0/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=3.0, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, ep

'/content/atlas_fn_workspace/Training_Runs/UniDet_GrandMaster_YOLO26_S5_PixelPerfect/weights/best.pt'

In [ ]:
# ============================================================
# 8. MANUSCRIPT-CORRECT SAM2-SMALL OFFLINE PSEUDO-LABELLING + NANO CONFIRM TRAINING
# ============================================================
# DockerRoot Cell 6 structure is preserved, but SAM2-Small is used instead of the old local sam3.pt.
# Output: compact YOLO26n-seg Confirm student checkpoint.

NANO_SEG_DIR = WORKSPACE / "dataset_nano_sam2_seg"
NANO_RUN_NAME = "NanoSpecialist_SAM2_Seg"

class PixelSpacingIndex:
    def __init__(self):
        self.spacing = {}
        self._load()
    def _load(self):
        # Prefer source ledger spacing from data-prep export.
        candidates = list(WORKSPACE.rglob("source_ledger_materialised.csv")) + list(WORKSPACE.rglob("source_ledger*.csv"))
        for p in candidates:
            try:
                df = pd.read_csv(p)
                if "filename" in df.columns and "spacing" in df.columns:
                    for _, r in df.iterrows():
                        try:
                            self.spacing[Path(str(r["filename"])).name] = float(r["spacing"])
                        except Exception:
                            pass
            except Exception:
                pass
    def get(self, filename, default=0.125):
        stem = Path(filename).stem
        for k in [Path(filename).name, stem + ".png", stem + ".jpg"]:
            if k in self.spacing:
                return self.spacing[k]
        return default

PIXELS = PixelSpacingIndex()

def clinical_enhance(img):
    if img is None or img.size == 0:
        return img
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    clean = cv2.bilateralFilter(gray, 5, 50, 50)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(clean)
    return cv2.merge([clahe] * 3)

def ensure_nano_yaml():
    y = {"path": str(NANO_SEG_DIR), "train": "images/train", "val": "images/val", "test": "images/test", "names": {0: "CSP", 1: "LV"}}
    (NANO_SEG_DIR / "data.yaml").write_text(yaml.safe_dump(y, sort_keys=False))
    return NANO_SEG_DIR / "data.yaml"

def polygon_from_mask(mask, w, h):
    if mask is None:
        return None
    mask_u8 = (mask > 0).astype(np.uint8)
    cnts, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    cnt = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(cnt) < 5:
        return None
    eps = max(1.0, 0.005 * cv2.arcLength(cnt, True))
    approx = cv2.approxPolyDP(cnt, eps, True).reshape(-1, 2).astype(float)
    if len(approx) < 3:
        return None
    approx[:, 0] = np.clip(approx[:, 0] / max(w, 1), 0, 1)
    approx[:, 1] = np.clip(approx[:, 1] / max(h, 1), 0, 1)
    return approx.reshape(-1).tolist()

def generate_nano_sam2_dataset():
    if (NANO_SEG_DIR / "data.yaml").exists() and not FORCE_REBUILD_NANO_SAM2_DATASET:
        log(f"reuse Nano SAM2 dataset: {NANO_SEG_DIR}")
        return ensure_nano_yaml()
    if NANO_SEG_DIR.exists() and FORCE_REBUILD_NANO_SAM2_DATASET:
        shutil.rmtree(NANO_SEG_DIR)
    for split in ["train", "val", "test"]:
        (NANO_SEG_DIR / f"images/{split}").mkdir(parents=True, exist_ok=True)
        (NANO_SEG_DIR / f"labels/{split}").mkdir(parents=True, exist_ok=True)

    try:
        sam = SAM(str(SAM2_SMALL))
    except Exception as e:
        record_stage("nano_sam2_dataset", "skipped", reason="SAM2 load failed", error=repr(e))
        return None

    audit_rows = []
    scales = {"Tight": 0.15, "Context": 0.45, "Regional": 1.10}
    total_imgs = sum(len(list((ROI_DIR / f"images/{s}").glob("*"))) for s in ["train", "val", "test"])
    if SMOKE_TEST:
        total_imgs = min(total_imgs, 60)

    with tqdm(total=total_imgs, desc="SAM2 pseudo masks", unit="img") as pbar:
        for split in ["train", "val", "test"]:
            imgs = sorted([p for p in (ROI_DIR / f"images/{split}").glob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}])
            if SMOKE_TEST:
                imgs = imgs[:20]
            for img_p in imgs:
                pbar.update(1)
                im0 = cv2.imread(str(img_p))
                if im0 is None:
                    continue
                h, w = im0.shape[:2]
                lbl_p = ROI_DIR / f"labels/{split}/{img_p.stem}.txt"
                if not lbl_p.exists() or not lbl_p.read_text().strip():
                    continue
                for idx, line in enumerate(lbl_p.read_text().splitlines()):
                    parts = line.split()
                    if len(parts) < 5:
                        continue
                    cls_id, nx, ny, nw, nh = int(float(parts[0])), *map(float, parts[1:5])
                    cx, cy, bw, bh = nx*w, ny*h, nw*w, nh*h
                    x1, y1, x2, y2 = int(cx-bw/2), int(cy-bh/2), int(cx+bw/2), int(cy+bh/2)
                    for scale_name, pad_r in scales.items():
                        pw, ph = int(bw*pad_r), int(bh*pad_r)
                        nx1, ny1 = max(0, x1-pw), max(0, y1-ph)
                        nx2, ny2 = min(w, x2+pw), min(h, y2+ph)
                        crop = im0[ny1:ny2, nx1:nx2]
                        if crop.size == 0:
                            continue
                        enhanced = clinical_enhance(crop)
                        box_prompt = [max(0, x1-nx1), max(0, y1-ny1), max(1, x2-nx1), max(1, y2-ny1)]
                        try:
                            res = sam(cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB), bboxes=[box_prompt], verbose=False)
                            mask = res[0].masks.data[0].cpu().numpy() if res and res[0].masks is not None else None
                            poly = polygon_from_mask(mask, enhanced.shape[1], enhanced.shape[0])
                            if not poly:
                                continue
                            out_name = f"{img_p.stem}_{idx}_{cls_id}_{scale_name}.png"
                            out_img = NANO_SEG_DIR / f"images/{split}/{out_name}"
                            out_lbl = NANO_SEG_DIR / f"labels/{split}/{out_name.replace('.png','.txt')}"
                            cv2.imwrite(str(out_img), enhanced)
                            out_lbl.write_text(f"{cls_id} " + " ".join(f"{v:.6f}" for v in poly))
                            audit_rows.append({"split": split, "source": str(img_p), "out": str(out_img), "class": cls_id, "scale": scale_name, "n_poly": len(poly)//2})
                        except Exception as e:
                            audit_rows.append({"split": split, "source": str(img_p), "class": cls_id, "scale": scale_name, "error": repr(e)})
                            continue
    ensure_nano_yaml()
    pd.DataFrame(audit_rows).to_csv(OUTPUT_ROOT / "tables/sam2_pseudo_mask_audit.csv", index=False)
    log(f"Nano SAM2 pseudo dataset complete | rows={len(audit_rows)}")
    return NANO_SEG_DIR / "data.yaml"

NANO_YAML = generate_nano_sam2_dataset()
CONFIRM_NANO_CKPT = None
if RUN_NANO_SAM2_CONFIRM_TRAINING and NANO_YAML:
    CONFIRM_NANO_CKPT = run_yolo_train(
        stage_name="confirm_nano_sam2_yolo26n_seg",
        base_model=YOLO26_SEG,
        data_yaml=NANO_YAML,
        project=RUNS_DIR,
        name=NANO_RUN_NAME,
        task="segment",
        epochs=1 if SMOKE_TEST else 30,
        imgsz=320,
        batch=32,
        optimizer="auto",
        cos_lr=True,
        workers=0,
        verbose=True,
        plots=True,
        amp=True,
    )
else:
    cand, _ = best_or_last(RUNS_DIR / NANO_RUN_NAME)
    CONFIRM_NANO_CKPT = str(cand) if cand.exists() else None

CONFIRM_NANO_CKPT


SAM2 pseudo masks:   0%|          | 0/3790 [00:00<?, ?img/s]

[atlas-train] Nano SAM2 pseudo dataset complete | rows=9919
[atlas-train] YOLO26 SPPF compatibility shim registered
[atlas-train] start training confirm_nano_sam2_yolo26n_seg | base=yolo26n-seg.pt, data=/content/atlas_fn_workspace/dataset_nano_sam2_seg/data.yaml, name=NanoSpecialist_SAM2_Seg
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/atlas_fn_workspace/dataset_nano_sam2_seg/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None,

'/content/atlas_fn_workspace/Training_Runs/NanoSpecialist_SAM2_Seg/weights/best.pt'

In [ ]:
# ============================================================
# 9. TRUE MEASURE STAGE EXTENSION — YOLO26n-seg head ellipse segmentation
# ============================================================
# This is the new implemented Measure-stage training product.
# It does not replace the manuscript geometric Measure pathway. It creates a separate
# segmentation checkpoint for later inference-impact analysis.

MEASURE_SEG_DIR = WORKSPACE / "dataset_measure_head_ellipse_seg"
MEASURE_RUN_NAME = "MeasureHeadEllipse_YOLO26n_Seg"

def ellipse_polygon_from_yolo_box(xc, yc, bw, bh, n=72):
    # normalized ellipse polygon within image coordinates
    angles = np.linspace(0, 2*np.pi, n, endpoint=False)
    xs = xc + (bw/2.0) * np.cos(angles)
    ys = yc + (bh/2.0) * np.sin(angles)
    pts = []
    for x, y in zip(xs, ys):
        pts.extend([float(np.clip(x, 0, 1)), float(np.clip(y, 0, 1))])
    return pts

def generate_measure_seg_dataset():
    if (MEASURE_SEG_DIR / "data.yaml").exists() and not FORCE_REBUILD_MEASURE_SEG_DATASET:
        log(f"reuse Measure segmentation dataset: {MEASURE_SEG_DIR}")
        return MEASURE_SEG_DIR / "data.yaml"
    if MEASURE_SEG_DIR.exists() and FORCE_REBUILD_MEASURE_SEG_DATASET:
        shutil.rmtree(MEASURE_SEG_DIR)
    for split in ["train", "val", "test"]:
        (MEASURE_SEG_DIR / f"images/{split}").mkdir(parents=True, exist_ok=True)
        (MEASURE_SEG_DIR / f"labels/{split}").mkdir(parents=True, exist_ok=True)

    rows = []
    for split in ["train", "val", "test"]:
        img_dir = BRAIN_DIR / "images" / split
        lbl_dir = BRAIN_DIR / "labels" / split
        imgs = sorted([p for p in img_dir.glob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}])
        if SMOKE_TEST:
            imgs = imgs[:50]
        for img_p in tqdm(imgs, desc=f"measure ellipse labels {split}"):
            out_img = MEASURE_SEG_DIR / f"images/{split}/{img_p.name}"
            if not out_img.exists():
                shutil.copy2(img_p, out_img)
            src_lbl = lbl_dir / f"{img_p.stem}.txt"
            out_lbl = MEASURE_SEG_DIR / f"labels/{split}/{img_p.stem}.txt"
            if not src_lbl.exists() or not src_lbl.read_text().strip():
                out_lbl.write_text("")
                rows.append({"split": split, "image": img_p.name, "status": "empty"})
                continue
            parts = src_lbl.read_text().splitlines()[0].split()
            if len(parts) < 5:
                out_lbl.write_text("")
                rows.append({"split": split, "image": img_p.name, "status": "bad_label"})
                continue
            _, xc, yc, bw, bh = map(float, parts[:5])
            poly = ellipse_polygon_from_yolo_box(xc, yc, bw, bh)
            out_lbl.write_text("0 " + " ".join(f"{v:.6f}" for v in poly))
            rows.append({"split": split, "image": img_p.name, "status": "ellipse", "points": len(poly)//2})

    yaml_data = {"path": str(MEASURE_SEG_DIR), "train": "images/train", "val": "images/val", "test": "images/test", "names": {0: "HeadEllipse"}}
    (MEASURE_SEG_DIR / "data.yaml").write_text(yaml.safe_dump(yaml_data, sort_keys=False))
    pd.DataFrame(rows).to_csv(OUTPUT_ROOT / "tables/measure_head_ellipse_dataset_audit.csv", index=False)
    log(f"Measure segmentation dataset ready | rows={len(rows)}")
    return MEASURE_SEG_DIR / "data.yaml"

MEASURE_YAML = generate_measure_seg_dataset()
MEASURE_CKPT = None
if RUN_TRUE_MEASURE_TRAINING and MEASURE_YAML:
    MEASURE_CKPT = run_yolo_train(
        stage_name="measure_head_ellipse_yolo26n_seg",
        base_model=YOLO26_SEG,
        data_yaml=MEASURE_YAML,
        project=RUNS_DIR,
        name=MEASURE_RUN_NAME,
        task="segment",
        epochs=1 if SMOKE_TEST else 30,
        imgsz=640,
        batch=16,
        optimizer="auto",
        cos_lr=True,
        workers=0,
        verbose=True,
        plots=True,
        amp=True,
    )
else:
    cand, _ = best_or_last(RUNS_DIR / MEASURE_RUN_NAME)
    MEASURE_CKPT = str(cand) if cand.exists() else None

MEASURE_CKPT


measure ellipse labels train:   0%|          | 0/2629 [00:00<?, ?it/s]

measure ellipse labels val:   0%|          | 0/575 [00:00<?, ?it/s]

measure ellipse labels test:   0%|          | 0/586 [00:00<?, ?it/s]

[atlas-train] Measure segmentation dataset ready | rows=3790
[atlas-train] YOLO26 SPPF compatibility shim registered
[atlas-train] start training measure_head_ellipse_yolo26n_seg | base=yolo26n-seg.pt, data=/content/atlas_fn_workspace/dataset_measure_head_ellipse_seg/data.yaml, name=MeasureHeadEllipse_YOLO26n_Seg
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/atlas_fn_workspace/dataset_measure_head_ellipse_seg/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchs

'/content/atlas_fn_workspace/Training_Runs/MeasureHeadEllipse_YOLO26n_Seg/weights/best.pt'

In [ ]:
# ============================================================
# 10. CHECKPOINT MANIFEST, TRAINING AUDIT, AND OPTIONAL VALIDATION
# ============================================================
checkpoint_manifest = [
    checkpoint_row("find", FIND_CKPT, "BMC_LOCKED: BrainLocator_YOLO26n_MuSGD"),
    checkpoint_row("find_bmc", FIND_CKPT, "BMC_LOCKED: BrainLocator_YOLO26n_MuSGD"),
    checkpoint_row("find_optimized", globals().get("OPTIMIZED_FIND_CKPT"), globals().get("OPTIMIZED_FIND_MODEL_LABEL") or "OPTIMISED_FIND"),
    checkpoint_row("confirm_grandmaster", CONFIRM_GRANDMASTER_CKPT, "UniDet_GrandMaster_YOLO26_S5_PixelPerfect"),
    checkpoint_row("confirm_nano_sam2", CONFIRM_NANO_CKPT, "NanoSpecialist_SAM2_Seg"),
    checkpoint_row("measure_true_seg", MEASURE_CKPT, "MeasureHeadEllipse_YOLO26n_Seg"),
    checkpoint_row("s1_federated", RUNS_DIR / "S1_FEDERATED.pt", "checkpoint_average"),
    checkpoint_row("s3_merged", RUNS_DIR / "S3_MERGED.pt", "task_vector_merge"),
]
CHECKPOINT_MANIFEST = pd.DataFrame(checkpoint_manifest)
CHECKPOINT_MANIFEST.to_csv(OUTPUT_ROOT / "tables/checkpoint_manifest.csv", index=False)

pd.DataFrame(STAGE_AUDIT).to_csv(OUTPUT_ROOT / "tables/training_stage_audit.csv", index=False)
pd.DataFrame(ISSUES).to_csv(OUTPUT_ROOT / "tables/nonfatal_training_issues.csv", index=False)

summary = {
    "created_utc": utc_now(),
    "checkpoints_total": len(CHECKPOINT_MANIFEST),
    "checkpoints_existing": int((CHECKPOINT_MANIFEST["status"] == "exists").sum()),
    "issues_total": len(ISSUES),
    "failed_stages": int(sum(1 for r in STAGE_AUDIT if r.get("status") in {"failed", "missing_checkpoint_after_train"})),
    "skipped_stages": int(sum(1 for r in STAGE_AUDIT if r.get("status") == "skipped")),
    "workspace": str(WORKSPACE),
    "runs_dir": str(RUNS_DIR),
}
save_json(OUTPUT_ROOT / "manifests/training_summary.json", summary)

print("CHECKPOINT MANIFEST")
display(CHECKPOINT_MANIFEST)
print("TRAINING SUMMARY")
print(json.dumps(summary, indent=2))


In [ ]:
# ============================================================
# 11. EXPORT CHECKPOINT PACKAGE FOR FINAL INFERENCE / MASTER-EVIDENCE NOTEBOOK
# ============================================================
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
package_name = f"ATLAS_FN_training_checkpoints_DockerRootReplica_{stamp}"
staging = EXPORT_ROOT / package_name
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True, exist_ok=True)

# Copy key checkpoints and metadata into a clean package.
def copy_if_exists(src, dst):
    src = Path(src) if src is not None else None
    dst = Path(dst)
    if src is None or not src.exists():
        return {"src": str(src), "dst": str(dst), "status": "missing"}
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return {"src": str(src), "dst": str(dst), "status": "copied", "bytes": dst.stat().st_size, "sha256": sha256_file(dst)}

copy_records = []
role_to_target = {
    "find": "checkpoints/find/best.pt",
    "find_bmc": "checkpoints/find_bmc/best.pt",
    "find_optimized": "checkpoints/find_optimized/best.pt",
    "confirm_grandmaster": "checkpoints/confirm_grandmaster/best.pt",
    "confirm_nano_sam2": "checkpoints/confirm_nano_sam2/best.pt",
    "measure_true_seg": "checkpoints/measure_true_seg/best.pt",
    "s1_federated": "checkpoints/derived/S1_FEDERATED.pt",
    "s3_merged": "checkpoints/derived/S3_MERGED.pt",
}
for _, r in CHECKPOINT_MANIFEST.iterrows():
    role = r["role"]
    if role in role_to_target:
        copy_records.append(copy_if_exists(r.get("path"), staging / role_to_target[role]))

# Include YAMLs and essential prepared-workspace ledgers for final inference.
essential_files = [
    BRAIN_YAML,
    STRUCT_YAML,
    ROI_YAML,
    NANO_SEG_DIR / "data.yaml",
    MEASURE_SEG_DIR / "data.yaml",
    OUTPUT_ROOT / "tables/checkpoint_manifest.csv",
    OUTPUT_ROOT / "tables/find_bmc_vs_optimised_training_contract.csv",
    OUTPUT_ROOT / "tables/find_hyperparameter_tuning_results.csv",
    OUTPUT_ROOT / "tables/training_stage_audit.csv",
    OUTPUT_ROOT / "tables/nonfatal_training_issues.csv",
    OUTPUT_ROOT / "tables/model_asset_manifest.csv",
    OUTPUT_ROOT / "tables/pre_training_dataset_inventory.csv",
    OUTPUT_ROOT / "manifests/runtime_info.json",
    OUTPUT_ROOT / "manifests/training_summary.json",
    OUTPUT_ROOT / "manifests/find_optimised_selection_summary.json",
]
for f in essential_files:
    if Path(f).exists():
        rel = Path("metadata") / Path(f).name
        copy_records.append(copy_if_exists(f, staging / rel))

# Include training result CSVs/plots lightly, not full raw run folders.
training_logs_dir = staging / "training_run_summaries"
training_logs_dir.mkdir(parents=True, exist_ok=True)
for run in [FIND_RUN_NAME, f"{RUN_NAME}_S5_PixelPerfect", NANO_RUN_NAME, MEASURE_RUN_NAME]:
    run_dir = RUNS_DIR / run
    if not run_dir.exists():
        continue
    out_dir = training_logs_dir / run
    out_dir.mkdir(parents=True, exist_ok=True)
    for pat in ["results.csv", "args.yaml", "results.png", "confusion_matrix.png", "labels.jpg"]:
        src = run_dir / pat
        if src.exists():
            copy_records.append(copy_if_exists(src, out_dir / pat))

# Write restore helper.
restore_helper = staging / "RESTORE_FOR_MASTER_EVIDENCE_NOTEBOOK.py"
restore_helper.write_text(r"""
from pathlib import Path
import shutil
# After extracting this package:
extracted = Path("/content/ATLAS_FN_training_checkpoints_DockerRootReplica_EXTRACTED")
workspace = Path("/content/atlas_fn_workspace")
workspace.mkdir(parents=True, exist_ok=True)
# Copy checkpoints into conventional locations used by the inference notebook.
mapping = {
    "checkpoints/find/best.pt": "Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt",
    "checkpoints/confirm_grandmaster/best.pt": "Training_Runs/UniDet_GrandMaster_YOLO26_S5_PixelPerfect/weights/best.pt",
    "checkpoints/confirm_nano_sam2/best.pt": "Training_Runs/NanoSpecialist_SAM2_Seg/weights/best.pt",
    "checkpoints/measure_true_seg/best.pt": "Training_Runs/MeasureHeadEllipse_YOLO26n_Seg/weights/best.pt",
    "checkpoints/derived/S1_FEDERATED.pt": "Training_Runs/S1_FEDERATED.pt",
    "checkpoints/derived/S3_MERGED.pt": "Training_Runs/S3_MERGED.pt",
}
for src_rel, dst_rel in mapping.items():
    src = extracted / src_rel
    dst = workspace / dst_rel
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
print("Checkpoint package restored into", workspace)
""")

# Package manifest.
package_manifest = {
    "created_utc": utc_now(),
    "package_name": package_name,
    "workspace": str(WORKSPACE),
    "copy_records": copy_records,
    "checkpoint_manifest": checkpoint_manifest,
    "training_summary": summary,
    "purpose": "Checkpoint package for ATLAS-FN final inference/master-evidence notebook",
}
save_json(staging / "PACKAGE_MANIFEST.json", package_manifest)

# File inventory with hashes.
file_inventory = []
for p in staging.rglob("*"):
    if p.is_file():
        file_inventory.append({"relative_path": str(p.relative_to(staging)), "bytes": p.stat().st_size, "sha256": sha256_file(p)})
save_json(staging / "PACKAGE_FILE_INVENTORY.json", file_inventory)

# Create archive.
archive_suffix = ".tar.gz" if EXPORT_COMPRESSION == "gz" else ".tar"
archive_path = EXPORT_ROOT / f"{package_name}{archive_suffix}"
if archive_path.exists():
    archive_path.unlink()
mode = "w:gz" if EXPORT_COMPRESSION == "gz" else "w"
with tarfile.open(archive_path, mode) as tar:
    tar.add(staging, arcname=staging.name)
archive_sha = sha256_file(archive_path)

archive_manifest = {
    "created_utc": utc_now(),
    "archive_path": str(archive_path),
    "archive_size_bytes": archive_path.stat().st_size,
    "archive_size_gb": archive_path.stat().st_size / (1024**3),
    "sha256": archive_sha,
    "package_name": package_name,
}
save_json(EXPORT_ROOT / f"{package_name}_ARCHIVE_MANIFEST.json", archive_manifest)

# Optional Drive copy.
drive_copy = None
if COPY_EXPORT_TO_DRIVE_IF_AVAILABLE and Path("/content/drive/MyDrive").exists():
    drive_dir = Path(DRIVE_EXPORT_DIR)
    drive_dir.mkdir(parents=True, exist_ok=True)
    drive_copy = drive_dir / archive_path.name
    shutil.copy2(archive_path, drive_copy)
    shutil.copy2(EXPORT_ROOT / f"{package_name}_ARCHIVE_MANIFEST.json", drive_dir / f"{package_name}_ARCHIVE_MANIFEST.json")

print("\n" + "="*90)
print("ATLAS-FN CHECKPOINT TRAINING EXPORT COMPLETE")
print("="*90)
print(json.dumps({**archive_manifest, "drive_copy": str(drive_copy) if drive_copy else None}, indent=2))
print("="*90)

try:
    from google.colab import files
    print("\nTo download manually from this Colab runtime:")
    print(f"files.download('{archive_path}')")
except Exception:
    pass



ATLAS-FN CHECKPOINT TRAINING EXPORT COMPLETE
{
  "created_utc": "2026-07-07T13:35:49+00:00",
  "archive_path": "/content/atlas_fn_training_exports/ATLAS_FN_training_checkpoints_DockerRootReplica_20260707_133539.tar.gz",
  "archive_size_bytes": 45622249,
  "archive_size_gb": 0.04248903039842844,
  "sha256": "f1f499315918e6ace904fd24a217ecf160b952e06241465395b1e4ef7f3e9538",
  "package_name": "ATLAS_FN_training_checkpoints_DockerRootReplica_20260707_133539",
  "drive_copy": "/content/drive/MyDrive/ATLAS_FN_exports/ATLAS_FN_training_checkpoints_DockerRootReplica_20260707_133539.tar.gz"
}

To download manually from this Colab runtime:
files.download('/content/atlas_fn_training_exports/ATLAS_FN_training_checkpoints_DockerRootReplica_20260707_133539.tar.gz')


## Expected completion signature

At the end of this notebook, the exported package should contain at least these checkpoint roles:

- `find` → `BrainLocator_YOLO26n_MuSGD/weights/best.pt`
- `confirm_grandmaster` → `UniDet_GrandMaster_YOLO26_S5_PixelPerfect/weights/best.pt`
- `confirm_nano_sam2` → `NanoSpecialist_SAM2_Seg/weights/best.pt`
- `measure_true_seg` → `MeasureHeadEllipse_YOLO26n_Seg/weights/best.pt`

The final inference/master-evidence notebook should be designed to consume this package plus the data-prep workspace package and should not retrain models.
